Imports

In [1]:
import re
from nltk.tokenize import word_tokenize
import nltk

import spacy

from sentence_transformers import SentenceTransformer, util
from thefuzz import fuzz
import torch, json, re

from transformers import pipeline

import json

import os


import time
from rdflib import Graph
from rdflib.plugins.sparql import parser
from rdflib.term import Literal
import rdflib

from sklearn.metrics import pairwise_distances
import csv
import numpy as np
import pandas as pd
from rdflib import URIRef

/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Take Namespaces

In [2]:
WD = rdflib.Namespace('http://www.wikidata.org/entity/')
WDT = rdflib.Namespace('http://www.wikidata.org/prop/direct/')
DDIS = rdflib.Namespace('http://ddis.ch/atai/')
RDFS = rdflib.namespace.RDFS
SCHEMA = rdflib.Namespace('http://schema.org/')

Load Given Embeddings

In [3]:
entity_emb = np.load(os.path.join('dataset/embeddings', 'entity_embeds.npy'))
relation_emb = np.load(os.path.join('dataset/embeddings', 'relation_embeds.npy'))

Load Graph

In [4]:
g = rdflib.Graph().parse(os.path.join( 'dataset', 'graph.nt'), format='turtle')
print(f"Graph loaded successfully with {len(g)} triples.")

Graph loaded successfully with 2413825 triples.


Ent->ID || ID->Ent Mappings

In [5]:
with open(os.path.join('dataset/embeddings', "entity_ids.del"), 'r') as ifile:
    ent2id = {rdflib.term.URIRef(ent): int(idx) for idx, ent in csv.reader(ifile, delimiter='\t')}
    id2ent = {v: k for k, v in ent2id.items()}
with open(os.path.join('dataset/embeddings', "relation_ids.del"), 'r') as ifile:
    rel2id = {rdflib.term.URIRef(rel): int(idx) for idx, rel in csv.reader(ifile, delimiter='\t')}
    id2rel = {v: k for k, v in rel2id.items()}

ent->lbl || lbl->ent

In [6]:
from rdflib import URIRef
from rdflib.namespace import RDFS, RDF

In [7]:
def build_label_maps(graph):
    """
    Build robust label–entity maps:

      - ent2lbl:        entity → plain label (string)
      - lbl2ents:       label → list of entity URIs
      - ent2lbl_type:   entity → "Label (Qxxxx)"
      - lbl2ent_type:   label → list of ("Label (Qxxxx)", entity URI)
    """
    P31 = URIRef("http://www.wikidata.org/prop/direct/P31")
    LABEL_PRED = RDFS.label

    ent2lbl = {}
    lbl2ents = {}
    ent2lbl_type = {}
    lbl2ent_type = {}

    for ent, lbl in graph.subject_objects(LABEL_PRED):
        label = str(lbl)
        ent2lbl[ent] = label
        lbl2ents.setdefault(label, []).append(ent)

        # --- get entity type ---
        etype = None
        for _, _, obj in graph.triples((ent, P31, None)):
            etype = obj.split("/")[-1]
            break
        if etype is None:
            for _, _, obj in graph.triples((ent, RDF.type, None)):
                etype = obj.split("/")[-1]
                break
        etype = etype or "unknown"

        label_type = f"{label} ({etype})"
        ent2lbl_type[ent] = label_type
        lbl2ent_type.setdefault(label, []).append((label_type, ent))

    return ent2lbl, lbl2ents, ent2lbl_type, lbl2ent_type

In [8]:
ent2lbl, lbl2ents, ent2lbl_type, lbl2ent_type = build_label_maps(g)




In [9]:
ent2lbl

{rdflib.term.URIRef('http://www.wikidata.org/entity/Q1138475'): 'Katsuhisa Namase',
 rdflib.term.URIRef('http://www.wikidata.org/entity/Q3573576'): 'Yves Corbeil',
 rdflib.term.URIRef('http://www.wikidata.org/entity/Q311303'): 'Adrian Pasdar',
 rdflib.term.URIRef('http://www.wikidata.org/entity/Q1494'): 'Mississippi',
 rdflib.term.URIRef('http://www.wikidata.org/entity/Q105441224'): 'Mr. Bachmann and His Class',
 rdflib.term.URIRef('http://www.wikidata.org/entity/Q3112667'): 'Hayden Schlossberg',
 rdflib.term.URIRef('http://www.wikidata.org/entity/Q23011556'): 'Mochaul Hiddlesten',
 rdflib.term.URIRef('http://www.wikidata.org/entity/Q65935802'): 'Corvus Glaive',
 rdflib.term.URIRef('http://www.wikidata.org/entity/Q1141706'): 'The Peacemaker',
 rdflib.term.URIRef('http://www.wikidata.org/entity/Q3295220'): 'Marthe Villalonga',
 rdflib.term.URIRef('http://www.wikidata.org/entity/Q2986219'): 'Heidi Swedberg',
 rdflib.term.URIRef('http://www.wikidata.org/entity/Q3154814'): 'Isabel Andrade'

In [10]:
lbl2ent_type['Othello']

[('Othello (Q3375722)',
  rdflib.term.URIRef('http://www.wikidata.org/entity/Q1966171')),
 ('Othello (Q11424)',
  rdflib.term.URIRef('http://www.wikidata.org/entity/Q2704454')),
 ('Othello (Q11424)',
  rdflib.term.URIRef('http://www.wikidata.org/entity/Q2634673')),
 ('Othello (Q11424)',
  rdflib.term.URIRef('http://www.wikidata.org/entity/Q2036869')),
 ('Othello (Q116476516)',
  rdflib.term.URIRef('http://www.wikidata.org/entity/Q26833'))]

Sentence Transformer Model to evaulate natural language

In [11]:
model = SentenceTransformer('all-MiniLM-L6-v2')

with open("predicate_label_map.json") as f:
    predicate_map = json.load(f)

predicate_labels = list(predicate_map.values())
predicate_embeddings = model.encode(predicate_labels, convert_to_tensor=True, normalize_embeddings=True)


nlp = spacy.load("en_core_web_sm")
ner_model = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [12]:
rel2lbl = {f"http://www.wikidata.org/prop/direct/{pid.upper()}": lbl for pid, lbl in predicate_map.items()}

# Create inverse mapping for lookup by label
lbl2rel = {lbl.lower(): f"http://www.wikidata.org/prop/direct/{pid.upper()}" for pid, lbl in predicate_map.items()}


In [13]:
predicate_labels = list(predicate_map.values())
predicate_embeddings = sbert_model.encode(predicate_labels, convert_to_tensor=True, normalize_embeddings=True)

# Category anchors for question-type consistency
category_prompts = {
    "PERSON": "This relation refers to a person or people.",
    "LOCATION": "This relation refers to a place or location.",
    "TIME": "This relation refers to a date, time or period.",
    "ATTRIBUTE": "This relation refers to a property, value, or description.",
    "MANNER": "This relation refers to a way or method."
}
cat_embs = sbert_model.encode(list(category_prompts.values()), convert_to_tensor=True, normalize_embeddings=True)
cat_labels = list(category_prompts.keys())

pred_cats = {}
for pid, label in predicate_map.items():
    emb = sbert_model.encode(label, convert_to_tensor=True, normalize_embeddings=True)
    sims = util.cos_sim(emb, cat_embs)[0]
    pred_cats[pid] = cat_labels[int(torch.argmax(sims))]

A Hybrid Relation Extraction Model Using Different Information

In [14]:
MOVIE_GAMMA_CORPUS = {
    # 🎞️ General film terms
    "film", "movie", "cinema", "motion picture", "feature", "short film",
    "screen", "screenplay", "script", "story", "plot", "synopsis",
    "character", "scene", "sequel", "prequel", "spin-off", "franchise",
    "series", "episode", "saga", "adaptation", "remake", "reboot",
    "soundtrack", "trailer", "poster", "credits", "budget", "box office",
    "runtime", "release", "premiere", "distribution", "studio", "production",
    "co-production", "producer", "executive producer", "director", "writer",
    "screenwriter", "cinematographer", "editor", "composer", "sound designer",
    "visual effects", "special effects", "set designer", "art director",
    "casting", "actor", "actress", "cast", "starring", "role", "character name",
    "lead role", "supporting role", "cameo", "voice actor", "voice acting",

    # 📅 Temporal & contextual metadata
    "release date", "release year", "premiere date", "filming location",
    "shooting location", "country of origin", "language", "production company",
    "distributor", "release country", "film festival", "award ceremony",
    "award", "nominated", "nomination", "won", "academy awards", "oscars",
    "cannes", "berlinale", "venice", "sundance", "bafta", "golden globe",

    # 🎭 Genres
    "action", "adventure", "animation", "biography", "comedy", "crime",
    "documentary", "drama", "family", "fantasy", "film noir", "history",
    "horror", "music", "musical", "mystery", "romance", "sci-fi", "science fiction",
    "short", "sports", "thriller", "war", "western",

    # 👩‍🎤 Cast & crew related
    "directed by", "written by", "produced by", "starring", "acted by",
    "music by", "cinematography by", "edited by", "distributed by",
    "production design", "art direction", "casting director", "screenwriter",
    "composer", "performer", "narrator",

    # 🏆 Awards & recognition
    "award", "oscar", "bafta", "golden globe", "cannes", "film festival",
    "nominated", "won", "best picture", "best actor", "best actress",
    "best director", "best screenplay", "best cinematography",

    # 🧠 Thematic / content attributes
    "based on", "adapted from", "inspired by", "set in", "takes place in",
    "filmed in", "filming", "setting", "time period", "era", "historical",
    "biographical", "fictional", "true story", "real events",

    # 📊 Metadata & analytics
    "rating", "imdb", "metacritic", "rotten tomatoes", "audience score",
    "critic score", "box office", "gross revenue", "budget", "profit", "earnings",
    "runtime", "duration", "minutes", "hours", "screen time",

    # 🌍 Cultural & language
    "language", "spoken in", "subtitles", "dubbed", "translation", "country",
    "nationality", "hollywood", "bollywood", "independent film", "foreign film",

    # 💡 Production / crew hierarchy
    "film crew", "crew member", "cinematography", "production", "editing",
    "soundtrack", "music composition", "director of photography", "lighting",
    "makeup artist", "costume designer", "visual effects supervisor",

    # 🧩 Narrative structure
    "plot", "storyline", "ending", "climax", "twist", "resolution",
    "theme", "motif", "symbolism", "dialogue", "screenplay", "narration",

    # 🧍‍♂️ Characters & relations
    "protagonist", "antagonist", "villain", "hero", "sidekick", "character arc",
    "voice", "role", "portrayed by", "appears in", "fictional character",

    # 💾 Technical
    "film format", "aspect ratio", "camera", "35mm", "digital", "color grading",
    "black and white", "hdr", "3d", "imax",

    # 🎥 Industry / business
    "production company", "film studio", "distributor", "streaming platform",
    "netflix", "amazon prime", "hbo", "disney", "apple tv", "hulu", "paramount",
    "warner bros", "universal", "sony pictures", "a24", "pixar", "dreamworks",
    "illuminations", "marvel", "dc comics", "lucasfilm", "20th century fox",

    # 🧩 Sequels / universes
    "series", "franchise", "universe", "cinematic universe", "crossover",
    "character appearance", "timeline", "shared universe", "connected films"
}


In [15]:
ner_pipeline = pipeline(
    "token-classification",
    model="Babelscape/wikineural-multilingual-ner",
    aggregation_strategy="simple"
)

Device set to use mps:0


In [16]:
from functools import lru_cache
import numpy as np

In [17]:
def keyword_score(q, label):
    """Lexical overlap using partial string match."""
    q_clean = re.sub(r"[^a-zA-Z0-9 ]", " ", q.lower())
    label_clean = re.sub(r"[^a-zA-Z0-9 ]", " ", label.lower())

    #TODO check
    date_phrases = ["come out", "released", "release date", "premiere", "premiered", "published", "publication", "time period", "day of week", "month of the year", "birthday"]
    if any(phrase in q_clean for phrase in date_phrases) and "date" in label_clean:
        # Boost score for date predicates when question contains date-related phrases
        base_score = fuzz.partial_ratio(q_clean, label_clean) / 100
        return min(1.0, base_score + 0.3)  # Add boost
    return fuzz.partial_ratio(q_clean, label_clean) / 100


def get_question_type(q):
    """Detect interrogative type (who/where/when/what/which/how)."""
    QUESTION_TYPES = {
        "who": "PERSON",
        "when": "TIME",
        "where": "LOCATION",
        "what": "ATTRIBUTE",
        "which": "ATTRIBUTE",
        "how": "MANNER"
    }
    q_lower = q.lower()
    for w, t in QUESTION_TYPES.items():
        if q_lower.startswith(w) or f" {w} " in q_lower:
            return t
    return None


def get_relation_candidates(question, alpha=0.45, beta=0.45, gamma=0.1, top_k=25):
    """
    Rank predicates by:
        α * semantic  +  β * lexical  +  γ * question-type consistency
    """
    q_type = get_question_type(question)
    q_emb = sbert_model.encode(question, convert_to_tensor=True, normalize_embeddings=True)
    cos_scores = util.cos_sim(q_emb, predicate_embeddings)[0]

    scored = []
    for idx, label in enumerate(predicate_labels):
        pid = list(predicate_map.keys())[idx]
        semantic = float(cos_scores[idx])
        lexical = keyword_score(question, label)
        base = alpha * semantic + beta * lexical

        # 🔹 Base boost (your original logic)
        boost = gamma if q_type and pred_cats.get(pid) == q_type else 0.0

        # 🔹 Additional gamma if movie-domain words appear in label
        if any(word in label.lower() for word in MOVIE_GAMMA_CORPUS):
            boost += gamma

        total = base + boost

        scored.append({
            "pid": pid,
            "label": label,
            "category": pred_cats.get(pid),
            "semantic": semantic,
            "lexical": lexical,
            "boost": boost,
            "score": total
        })

    scored = sorted(scored, key=lambda x: x["score"], reverse=True)[:top_k]
    return scored



def extract_relations_and_entities(question, alpha=0.5, beta=0.4, gamma=0.1):
    """
    Full pipeline:
    - Extract entities with precedence: 'quotes' → NER → spaCy/PROPN
    - Rank relation predicates (semantic + lexical + question-type)
    - Replace entity mentions with 'X' for relation scoring only
    """
    doc = nlp(question)
    entities = []

    # --- 1️⃣ Transformer-based NER (main extractor) ---
    ner_results = ner_pipeline(question)
    for ent in ner_results:
        text = ent["word"].strip("'\"")
        label = ent["entity_group"].upper()
        start, end = ent["start"], ent["end"]
        entities.append({"text": text, "label": label, "start": start, "end": end})

    # --- 2️⃣ Quoted entities fallback (if transformer found none) ---
    if not entities:
        pattern = r"'([^']+)'|\"([^\"]+)\"|‘([^’]+)’|“([^”]+)”"
        for match in re.finditer(pattern, question):
            text = next(g for g in match.groups() if g)
            start, end = match.span()
            entities.append({"text": text, "label": "QUOTED", "start": start, "end": end})

    # --- 3️⃣ spaCy fallback (if still none) ---
    if not entities:
        for ent in doc.ents:
            entities.append({
                "text": ent.text.strip().strip("'\""),
                "label": ent.label_.upper(),
                "start": ent.start_char,
                "end": ent.end_char
            })
        if not entities:  # proper noun heuristic
            current = []
            for token in doc:
                if token.pos_ == "PROPN":
                    current.append(token)
                elif current:
                    name = " ".join(t.text for t in current)
                    start = current[0].idx
                    end = current[-1].idx + len(current[-1])
                    entities.append({"text": name, "label": "PROPN", "start": start, "end": end})
                    current = []
            if current:
                name = " ".join(t.text for t in current)
                start = current[0].idx
                end = current[-1].idx + len(current[-1])
                entities.append({"text": name, "label": "PROPN", "start": start, "end": end})

    # --- 4️⃣ Deduplicate entities ---
    seen = set()
    entities = [e for e in entities if not (e["text"] in seen or seen.add(e["text"]))]

    # --- 5️⃣ Normalize question (replace entity mentions with 'X') ---
    normalized_question = question
    for ent in entities:
        ent_text = re.escape(ent["text"])
        normalized_question = re.sub(ent_text, "X", normalized_question, flags=re.IGNORECASE)

    # --- 6️⃣ Relation candidates (semantic–lexical–contextual) ---
    ranked_rels = get_relation_candidates(normalized_question, alpha=alpha, beta=beta, gamma=gamma)
    relation_candidates = [r["label"] for r in ranked_rels]
    scores = [r["score"] for r in ranked_rels]

    # --- 6.5️⃣ Entity type disambiguation with context rewriting ---
        # Small set of anchor type words to stabilize the short-label extraction
    _ANCHORS = [
        # Creative works / media
        "film", "movie", "short film", "television series", "episode",
        "play", "theatrical work", "opera", "musical", "ballet",
        "book", "novel", "poem", "literary work", "essay", "comic book",
        "song", "album", "music recording", "painting", "artwork",

        # People / professions / roles
        "person", "actor", "director", "writer", "composer", "artist",  "politician",

        # Miscellaneous entities / others
        "organization", "company", "place", "city", "country",
        "video game", "fictional character", "historical event"
    ]
    _anchors_emb = sbert_model.encode(_ANCHORS, convert_to_tensor=True, normalize_embeddings=True)

    @lru_cache(maxsize=4096)
    def _get_type_label_cached(type_id: str):
        """Return human label for a wikidata type id (e.g., Q11424 -> 'film')."""
        t_uri = URIRef(f"http://www.wikidata.org/entity/{type_id}")
        for _, _, lab in g.triples((t_uri, RDFS.label, None)):
            return str(lab)
        return type_id  # fallback

    @lru_cache(maxsize=8192)
    def _get_entity_desc_cached(uri_str: str):
        """Return schema:description (first sentence) if any."""
        d = ""
        for _, _, dsc in g.triples((URIRef(uri_str), URIRef("http://schema.org/description"), None)):
            d = str(dsc)
            break
        if not d:
            return ""
        # take first sentence
        first = d.split(".")[0]
        return first

    def _semantic_short_label(type_label: str, desc_first: str, q_emb):
        """
        Build 1–2 word semantic type phrase from type/description using SBERT.
        Not rule-based: pick the candidate phrase with best semantic fit to question
        and anchors (stability).
        """
        candidates = set()

        # seed with the type label itself
        if type_label:
            candidates.add(type_label.strip())

        # mine a few short candidates from the description's first sentence
        if desc_first:
            doc_desc = nlp(desc_first)
            # noun chunks (short)
            for nc in doc_desc.noun_chunks:
                phrase = nc.text.strip()
                if 1 <= len(phrase.split()) <= 3:
                    candidates.add(phrase)
            # first tokens bigrams/trigrams for extra coverage
            toks = [t.text for t in doc_desc if not t.is_space][:10]
            for n in (1, 2, 3):
                for i in range(len(toks) - n + 1):
                    phrase = " ".join(toks[i:i+n]).strip()
                    if 1 <= len(phrase.split()) <= 3:
                        candidates.add(phrase)

        # final fallback
        if not candidates:
            candidates.add(type_label or "entity")

        cand_list = list(candidates)
        cand_emb = sbert_model.encode(cand_list, convert_to_tensor=True, normalize_embeddings=True)

        # similarity to question
        q_sim = util.cos_sim(q_emb, cand_emb)[0]  # shape [num_cands]
        # similarity to anchors (take max over anchors for each candidate)
        a_sim = util.cos_sim(cand_emb, _anchors_emb).max(dim=1).values

        # combined score (all semantic)
        # weights: more weight to anchors to prefer clean type-words; tune if needed
        score = 0.6 * q_sim + 0.4 * a_sim
        best_idx = int(score.argmax().item())
        return cand_list[best_idx]

    entity_enriched = []
    for e in entities:
        label_text = e["text"].strip()
        all_uris, all_types, all_type_labels, all_rewritten, cand_scores = [], [], [], [], []

        if label_text in lbl2ent_type:
            candidates = lbl2ent_type[label_text]  # list of (label_with_type, URIRef)
            q_emb = sbert_model.encode(question, convert_to_tensor=True, normalize_embeddings=True)

            # Build candidates with short semantic type phrases
            rewritten_variants = []
            for label_type, uri in candidates:
                type_id = label_type.split("(")[-1].rstrip(")")
                uri_str = str(uri)
                type_label = _get_type_label_cached(type_id)
                desc_first = _get_entity_desc_cached(uri_str)

                short_type = _semantic_short_label(type_label, desc_first, q_emb)

                # rewrite for context scoring (kept simple and non-brittle)
                # prefer "the <short_type> <label_text>" if short_type is not 'person' etc.
                if short_type.lower() in {"person"}:
                    rewritten = question.replace(label_text, f"{label_text} ({short_type})")
                else:
                    rewritten = question.replace(label_text, f"the {short_type} {label_text}")

                rewritten_variants.append((uri_str, type_id, type_label, short_type, rewritten))

            # Rank by semantic similarity between original question and rewritten variants
            rew_texts = [rw[-1] for rw in rewritten_variants]
            rew_emb = sbert_model.encode(rew_texts, convert_to_tensor=True, normalize_embeddings=True)
            sim_scores = util.cos_sim(q_emb, rew_emb)[0].cpu().numpy()

            # sort by descending similarity
            ranked_idx = np.argsort(sim_scores)[::-1]
            for i in ranked_idx:
                uri_str, type_id, type_label, short_type, rewritten = rewritten_variants[i]
                all_uris.append(uri_str)
                all_types.append(type_id)
                all_type_labels.append(type_label or type_id)
                all_rewritten.append(rewritten)
                cand_scores.append(float(sim_scores[i]))

            # (optional) debug print
            print(f"\n🔍 Entity '{label_text}' disambiguation:")
            for i, sc in enumerate(cand_scores, 1):
                print(f"   Option {i}: {all_type_labels[i-1]} → {all_rewritten[i-1]} (score={sc:.3f})")
            print(f"   → Selected: {all_rewritten[0]} [{all_types[0]}]")

        else:
            # not found in KG
            all_uris, all_types, all_type_labels, all_rewritten = [], [], [], []

        entity_enriched.append({
            **e,
            "uri": all_uris if all_uris else [None],
            "type": all_types if all_types else ["#N/A"],
            "type_labels": all_type_labels if all_type_labels else ["#N/A"],
            "rewritten": all_rewritten if all_rewritten else [question]
        })


    print(
        "entity_candidates:", entity_enriched,
        "ranked_relations:", ranked_rels,
        "relation_candidates:", relation_candidates,
        "question_type:", get_question_type(question),
        "score:", scores
    )

    # --- 7️⃣ Return unified result (with entity types) ---
    return {
        "entity_candidates": entity_enriched,  # includes URIs, types, and type_labels
        "ranked_relations": ranked_rels,
        "relation_candidates": relation_candidates,
        "question_type": get_question_type(question),
        "score": scores
    }




In [18]:
q = "Who is the creator of the Othello movie?"
extract_relations_and_entities(q)



🔍 Entity 'Othello' disambiguation:
   Option 1: film → Who is the creator of the the film Othello movie? (score=0.991)
   Option 2: film → Who is the creator of the the film Othello movie? (score=0.991)
   Option 3: film → Who is the creator of the the film Othello movie? (score=0.991)
   Option 4: Q116476516 → Who is the creator of the the theatrical Othello movie? (score=0.961)
   Option 5: Q3375722 → Who is the creator of the the Shakespeare's Othello Othello movie? (score=0.916)
   → Selected: Who is the creator of the the film Othello movie? [Q11424]
entity_candidates: [{'text': 'Othello', 'label': 'PER', 'start': 26, 'end': 33, 'uri': ['http://www.wikidata.org/entity/Q2036869', 'http://www.wikidata.org/entity/Q2634673', 'http://www.wikidata.org/entity/Q2704454', 'http://www.wikidata.org/entity/Q26833', 'http://www.wikidata.org/entity/Q1966171'], 'type': ['Q11424', 'Q11424', 'Q11424', 'Q116476516', 'Q3375722'], 'type_labels': ['film', 'film', 'film', 'Q116476516', 'Q3375722'], 'r

{'entity_candidates': [{'text': 'Othello',
   'label': 'PER',
   'start': 26,
   'end': 33,
   'uri': ['http://www.wikidata.org/entity/Q2036869',
    'http://www.wikidata.org/entity/Q2634673',
    'http://www.wikidata.org/entity/Q2704454',
    'http://www.wikidata.org/entity/Q26833',
    'http://www.wikidata.org/entity/Q1966171'],
   'type': ['Q11424', 'Q11424', 'Q11424', 'Q116476516', 'Q3375722'],
   'type_labels': ['film', 'film', 'film', 'Q116476516', 'Q3375722'],
   'rewritten': ['Who is the creator of the the film Othello movie?',
    'Who is the creator of the the film Othello movie?',
    'Who is the creator of the the film Othello movie?',
    'Who is the creator of the the theatrical Othello movie?',
    "Who is the creator of the the Shakespeare's Othello Othello movie?"]}],
 'ranked_relations': [{'pid': 'p170',
   'label': 'creator',
   'category': 'PERSON',
   'semantic': 0.4340742528438568,
   'lexical': 1.0,
   'boost': 0.1,
   'score': 0.7170371264219284},
  {'pid': 'p31

In [19]:
with open("predicates_list.txt", "r") as f:
    predicate_list = [line.strip() for line in f if line.strip()]


In [20]:
_predicate_labels = list(predicate_map.values())
_predicate_ids = list(predicate_map.keys())
_predicate_embs = sbert_model.encode(_predicate_labels, convert_to_tensor=True, normalize_embeddings=True)

In [21]:

def find_exact_predicate(relation_phrase):
    """
    Return (pid, label) if the relation exactly matches a known predicate label.
    Otherwise, return (None, None).
    """
    if not relation_phrase:
        return None, None
    rel = relation_phrase.lower().strip()
    for pid, label in predicate_map.items():
        if rel == label.lower():
            return pid, label
    return None, None


def _hybrid_best_predicate(relation_phrase, alpha=0.4, beta=0.6):
    """
    Fallback mapping: semantic (MiniLM) + lexical similarity to predicate labels.
    Returns (pid, label, score).
    """
    if not relation_phrase:
        return None, None, 0.0

    q_emb = sbert_model.encode(relation_phrase, convert_to_tensor=True, normalize_embeddings=True)
    cos_scores = util.cos_sim(q_emb, _predicate_embs)[0]

    best_idx, best_score = None, -1.0
    for idx, label in enumerate(_predicate_labels):
        semantic = float(cos_scores[idx])
        lexical = fuzz.partial_ratio(relation_phrase.lower(), label.lower()) / 100
        score = alpha * semantic + beta * lexical
        if score > best_score:
            best_idx, best_score = idx, score

    if best_idx is None:
        return None, None, 0.0

    return _predicate_ids[best_idx], _predicate_labels[best_idx], best_score


def build_sparql_query_auto(parsed):
    """
    Build a SPARQL query compatible with your top-k pipeline.
    Input:
        parsed = {"relation_phrase": str, "entity_phrase": str}
    Logic:
        1. Try exact predicate match.
        2. Fallback to hybrid semantic+lexical match.
        3. Generate SPARQL in your example style.
    """
    relation = parsed.get("relation_phrase")
    entity = parsed.get("entity_phrase")

    if not relation or not entity:
        print("Missing relation or entity in parsed question.")
        return None

    # Exact match first
    pid, label = find_exact_predicate(relation)

    # Hybrid fallback
    if not pid:
        pid, label, score = _hybrid_best_predicate(relation)
        if not pid:
            print(f"No predicate found for relation '{relation}'")
            return None

    # Safe entity escaping
    entity_safe = entity.replace('"', '\\"').strip()

    # Variable base name
    var_base = re.sub(r"[^a-zA-Z0-9_]", "", label.lower().replace(" ", "_")) or "answer"

    # Prefixes (added rdfs)
    prefixes = """
PREFIX ddis: <http://ddis.ch/atai/>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX schema: <http://schema.org/>
""".strip()

    # Query template (kept identical to your structure)
    # Predicates that return literal values (dates, numbers, strings) rather than entities
    literal_predicates = {"p577", "p569", "p570", "p571", "p580", "p582", "p585"}  # dates
    # Add more numeric/string literal predicates as needed
    
    pid_lower = pid.lower()
    
    # Check if this predicate returns a literal value
    if pid_lower in literal_predicates or "date" in label.lower() or "time" in label.lower():
        # For literal values (dates, numbers, etc.), return the literal directly
        query = f"""

{prefixes}
SELECT ?{var_base}Value WHERE {{
  ?movieItem rdfs:label "{entity_safe}" .
  ?movieItem wdt:{pid.upper()} ?{var_base}Value .
}}
LIMIT 10
""".strip()
    else:
        # For entity values, get the label
        query = f"""
{prefixes}
SELECT ?{var_base}Label ?{var_base}Item WHERE {{
  ?movieItem rdfs:label "{entity_safe}" .
  ?movieItem wdt:{pid.upper()} ?{var_base}Item .
  ?{var_base}Item rdfs:label ?{var_base}Label .
}}
LIMIT 10
""".strip()

    return {
        "query": query,
        "matched_predicate": label,
        "pid": pid
    }


In [22]:
def rank_relation_candidates(relation_strings, *_, **__):
    """
    Pass-through ranker.

    Input:
        relation_strings : list[str]
            Already-ranked relation labels (e.g., from parsed['relation_candidates']).

    Output:
        list[str] - same order, unchanged.
    """
    if not relation_strings:
        return []
    return list(relation_strings)

SPARQL QUERY BUILDER

In [23]:
def build_topk_sparql_queries(parsed, k=10):
    """
    Generate up to k SPARQL queries ranked by relation probability.
    Stops early if no relation/entity extracted.
    """
    relations = parsed.get("relation_candidates", [])
    entities = parsed.get("entity_candidates", [])

    if not relations or not entities:
        print("Missing relations or entities.")
        return []

    ranked_relations = rank_relation_candidates(relations)
    print(f"Ranked relations: {ranked_relations}")
    entity_phrase = entities[0]["text"]  # use the most confident entity for now

    queries = []
    for relation in ranked_relations[:k]:
        sparql_input = {
            "relation_phrase": relation,
            "entity_phrase": entity_phrase
        }
        result = build_sparql_query_auto(sparql_input)
        if result:
            result["relation_candidate"] = relation
            queries.append(result)

    return queries


In [24]:
message = ["Who is the screenwriter of The Masked Gang: Cyprus?"]

In [25]:
from rdflib.term import Literal
from rdflib.plugins.sparql import parser

def on_query(message : str, g):
    try:
        parser.parseQuery(message)
        results = g.query(message)
    
    except Exception as e:
        return [f"Error executing query: {e}"]
    answer = []
    for row in results:
        for var in row.labels:
            val = row[var]
            if isinstance(val, Literal):
                answer.append(val.toPython())
    if not answer:
        return ["Query executed, but no matching results were found."]
    return answer

for q in message:
    parsed = extract_relations_and_entities(q)
    print("q:",q)
    queries = build_topk_sparql_queries(parsed, k=100)

    print(f"\nQuestion: {q}")
    print(f"Extracted relations: {parsed['relation_candidates']}")
    print("Generated SPARQL queries:")
    for i, qdict in enumerate(queries, start=1):
        print(f"\n--- Query {i} ({qdict['relation_candidate']}) ---")
        print(qdict["query"])
        print(on_query(qdict["query"], g))

entity_candidates: [{'text': 'The Masked Gang : Cyprus', 'label': 'MISC', 'start': 27, 'end': 50, 'uri': [None], 'type': ['#N/A'], 'type_labels': ['#N/A'], 'rewritten': ['Who is the screenwriter of The Masked Gang: Cyprus?']}] ranked_relations: [{'pid': 'p58', 'label': 'screenwriter', 'category': 'PERSON', 'semantic': 0.5254210233688354, 'lexical': 1.0, 'boost': 0.2, 'score': 0.8627105116844178}, {'pid': 'p2515', 'label': 'costume designer', 'category': 'PERSON', 'semantic': 0.3696964979171753, 'lexical': 0.47, 'boost': 0.2, 'score': 0.5728482489585877}, {'pid': 'p3092', 'label': 'film crew member', 'category': 'PERSON', 'semantic': 0.3846769332885742, 'lexical': 0.44, 'boost': 0.2, 'score': 0.5683384666442872}, {'pid': 'p1431', 'label': 'executive producer', 'category': 'PERSON', 'semantic': 0.42255550622940063, 'lexical': 0.39, 'boost': 0.2, 'score': 0.5672777531147004}, {'pid': 'p725', 'label': 'voice actor', 'category': 'PERSON', 'semantic': 0.36710822582244873, 'lexical': 0.45, 'b

In [26]:
def find_uri_by_label(label_dict, label, sbert_model=None, ent2lbl_embeds=None, threshold=80):
    """
    Finds the URI in label_dict whose label best matches the given text.
    Matching priority:
      1. Exact (case-insensitive)
      2. Bidirectional substring match
      3. Fuzzy (edit-distance-based)
      4. Optional SBERT semantic similarity fallback if given
    Returns URI if found, else None.
    """
    label_lower = label.lower().strip()

    # 1️⃣ Exact match
    for uri, lbl in label_dict.items():
        if lbl.lower() == label_lower:
            return uri

    # 2️⃣ Bidirectional substring match (query contains KG label OR vice versa)
    for uri, lbl in label_dict.items():
        lbl_lower = lbl.lower()
        if (lbl_lower in label_lower or label_lower in lbl_lower) and len(lbl_lower) > 3:
            return uri

    # 3️⃣ Fuzzy match (edit distance)
    best_uri = None
    best_ratio = 0
    for uri, lbl in label_dict.items():
        ratio = fuzz.token_sort_ratio(label_lower, lbl.lower())  # better for swapped words
        if ratio > best_ratio:
            best_ratio = ratio
            best_uri = uri
    if best_ratio >= threshold:
        print(f"[FUZZY MATCH] '{label}' ≈ '{label_dict[best_uri]}' (score={best_ratio})")
        return best_uri

    # 4️⃣ SBERT semantic similarity fallback (if model + embeddings provided)
    if sbert_model is not None:
        label_emb = sbert_model.encode(label, normalize_embeddings=True)
        best_sim, best_uri = 0.0, None
        for uri, lbl in label_dict.items():
            emb2 = ent2lbl_embeds.get(lbl)
            if emb2 is None:
                continue
            sim = float(util.cos_sim(label_emb, emb2))
            if sim > best_sim:
                best_sim = sim
                best_uri = uri
        if best_sim > 0.7:
            print(f"[SBERT MATCH] '{label}' ≈ '{label_dict[best_uri]}' (cos={best_sim:.2f})")
            return best_uri

    print(f"[WARN] No good match found for '{label}' (best fuzzy={best_ratio:.1f})")
    return None


In [27]:
def get_entity_type(entity_uri, graph):
    """
    Finds the type (wdt:P31 / rdf:type) of an entity in the RDF graph.
    Returns the short form like 'Q5' or 'Q11424', or 'unknown' if not found.
    """
    # Common Wikidata predicates for type
    P31 = URIRef("http://www.wikidata.org/prop/direct/P31")
    RDF_TYPE = URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")

    # Try P31 first (preferred)
    for _, _, obj in graph.triples((URIRef(entity_uri), P31, None)):
        return obj.split("/")[-1]  # e.g., Q5, Q11424

    # Fallback: try rdf:type
    for _, _, obj in graph.triples((URIRef(entity_uri), RDF_TYPE, None)):
        return obj.split("/")[-1]

    return "unknown"

In [28]:
from sklearn.metrics.pairwise import cosine_similarity


In [29]:
def pairwise_cosine_ordered(lhs, entity_emb, top=25):
    """
    Compute cosine distances between lhs vector and all entity embeddings.
    Return both distances and sorted indices (ascending = most similar first).
    """

    # Normalize vectors for cosine similarity
    lhs_norm = lhs / (np.linalg.norm(lhs) + 1e-10)
    entity_norm = entity_emb / (np.linalg.norm(entity_emb, axis=1, keepdims=True) + 1e-10)

    # Cosine similarity
    cosine_sim = np.dot(entity_norm, lhs_norm)
    # Convert to distance (1 - similarity)
    cosine_dist = 1 - cosine_sim

    # Get ordered indices by similarity (largest sim → smallest distance)
    ordered_idx = np.argsort(cosine_dist)[:top]

    # Return in same format as pairwise_distances
    return cosine_dist, ordered_idx


In [30]:
def embedding_answer(ent, rel, top=10):
    """
    Given an entity label (ent) and relation label (rel),
    compute the most plausible tail entities using TransE-style embedding reasoning.
    """

    # Find the corresponding URIs
    ent_uri = find_uri_by_label(ent2lbl, ent["text"])
    rel_uri = find_uri_by_label(rel2lbl, rel)
    if not ent_uri or not rel_uri:
        print(f"Could not find entity '{ent}' or relation '{rel}' in label maps.")
        return None

    # Convert to embedding indices
    try:
        h_id = ent2id[ent_uri]
        r_id = rel2id[rdflib.term.URIRef(rel_uri)]
    except KeyError as e:
        print(f"Missing embedding index for {e}")
        return None

    # Compute the TransE-style translation vector
    lhs = entity_emb[h_id] + relation_emb[r_id]
    print(h_id, r_id)

    # Compute distances to all entities
    dist = pairwise_distances(lhs.reshape(1, -1), entity_emb).reshape(-1)
    best = np.argsort(dist)[:top]

    # Prepare readable results
    rows = []
    for rank, idx in enumerate(best, 1):
        try:
            uri = id2ent[idx]
            label = ent2lbl.get(uri, "N/A")
            uri_short = uri[len(WD):] if str(uri).startswith(str(WD)) else str(uri)
            rows.append({
                "Rank": rank,
                "Entity": uri_short,
                "Label": label,
                "URI": uri,
                "Distance": round(float(dist[idx]), 6)
            })
        except Exception as e:
            print(f"Skipped index {idx}: {e}")
            continue

    df = pd.DataFrame(rows, columns=["Rank", "Entity", "Label", "URI", "Distance"])

    # ✅ Find first non-N/A label as top embedding answer
    valid_rows = df[df["Label"] != "N/A"]
    if not valid_rows.empty:
        top_label = valid_rows.iloc[0]["Label"]
        top_uri = valid_rows.iloc[0]["URI"]
        top_type = get_entity_type(top_uri, g)
        embedding_text = f"The answer suggested by embeddings is: {top_label} (type: {top_type})"
        print(top_label)
    else:
        embedding_text = "No embedding-based answer found."

    return df, embedding_text

# ==========================================================
# 🎬  Movie Recommendation Logic with Detailed Debug Prints
# ==========================================================

In [31]:
import curses
from collections import defaultdict

In [32]:
def find_movie_uri_by_label(label_dict, label, graph=g):
    """
    Specialized resolver for recommendation logic.
    Finds the URI for a movie label, prioritizing entities whose 'instance of' (P31)
    indicates a film/movie/animation. Assumes labels do NOT contain parentheses or suffixes.
    """
    label_lower = label.strip().lower()
    P31 = URIRef("http://www.wikidata.org/prop/direct/P31")
    film_keywords = {"film", "movie", "feature film", "animated", "television film"}
    candidates = []

    # --- Step 1: Find all exact or case-insensitive label matches ---
    for uri, lbl in label_dict.items():
        if lbl.lower() == label_lower:
            candidates.append(uri)

    if not candidates:
        # Optional fuzzy fallback
        best_uri, best_ratio = None, 0
        for uri, lbl in label_dict.items():
            ratio = fuzz.ratio(label_lower, lbl.lower())
            if ratio > best_ratio:
                best_ratio, best_uri = ratio, uri
        if best_uri and best_ratio > 85:
            candidates.append(best_uri)

    if not candidates:
        print(f"    ⚠️ No match found for '{label}'")
        return None

    # --- Step 2: Prioritize film-type entities ---
    film_candidates = []
    for uri in candidates:
        for _, _, obj in graph.triples((uri, P31, None)):
            type_lbl = label_dict.get(obj, "").lower()
            if any(k in type_lbl for k in film_keywords):
                film_candidates.append(uri)
                break

    if film_candidates:
        # Choose the first film-type entity
        best_uri = film_candidates[0]
        chosen_lbl = label_dict.get(best_uri, best_uri.split("/")[-1])
        print(f"      🎬 Film-type match prioritized for '{label}': {chosen_lbl} ({best_uri.split('/')[-1]})")
        return best_uri

    # --- Step 3: Fallback: pick the first exact match ---
    best_uri = candidates[0]
    chosen_lbl = label_dict.get(best_uri, best_uri.split("/")[-1])
    print(f"      ⚠️ Non-film match (fallback) for '{label}': {chosen_lbl} ({best_uri.split('/')[-1]})")
    return best_uri


In [33]:
# ===============================================================
# 1️⃣  MEANINGFUL RELATIONS (direct + semantically relevant)
# ===============================================================
meaningful_relations = {
    "P31", "P136", "P179", "P272", "P144", "P155", "P156",
    "P162", "P86", "P57", "P58", "P161", "P725",
    "P1877", "P364", "P361"
}


def get_fast_onehop_metadata(movie_uri, graph, ent2lbl, meaningful_relations, limit=8):
    """
    NEW: Extremely fast 1-hop metadata extraction.
    Only meaningful relations, only URI objects, no fuzzy matching.
    """
    uri = URIRef(movie_uri)
    meta = {rel: set() for rel in meaningful_relations}

    for _, rel, obj in graph.triples((uri, None, None)):
        pid = rel.split("/")[-1].upper()
        if pid not in meaningful_relations:
            continue
        if not isinstance(obj, URIRef):
            continue
        lbl = ent2lbl.get(obj)
        if lbl:
            if len(meta[pid]) < limit:
                meta[pid].add(lbl)

    return meta



In [34]:
def compute_movie_centroid(meta, sbert_model):
    """
    NEW: Compute a single SBERT centroid vector for each movie's metadata.
    Replaces all old SBERT-per-label functions.
    """
    items = [lbl for vals in meta.values() for lbl in vals]
    if not items:
        return None
    embs = sbert_model.encode(items, normalize_embeddings=True)
    return np.mean(embs, axis=0)


In [35]:
def tiny_transe_bonus(lbl_a, lbl_b, ent2id, entity_emb):
    """
    NEW: Tiny optional bonus using TransE similarity (fast, safe).
    No global scans.
    """
    try:
        id1 = ent2id.get(lbl_a)
        id2 = ent2id.get(lbl_b)
        if id1 is None or id2 is None:
            return 0.0
        v1, v2 = entity_emb[id1], entity_emb[id2]
        denom = (np.linalg.norm(v1) * np.linalg.norm(v2)) + 1e-10
        return float(np.dot(v1, v2) / denom)
    except:
        return 0.0


In [36]:
def merge_metadata_list(meta_list):
    """
    NEW: Merge metadata from multiple movies into one candidate pool.
    """
    merged = set()
    for meta in meta_list:
        for vals in meta.values():
            merged.update(vals)
    return list(merged)


In [37]:
def rank_metadata_candidates(merged_labels, movie_centroids, sbert_model, top_k=10):
    """
    NEW: Score candidates with SBERT similarity to each movie centroid.
    Replaces all heavy recommendation functions.
    """
    if not merged_labels:
        return []

    label_embs = sbert_model.encode(merged_labels, normalize_embeddings=True)
    scored = []

    for lbl, emb in zip(merged_labels, label_embs):
        sims = [float(util.cos_sim(emb, mc)) for mc in movie_centroids]
        score = sum(sims) / len(sims)
        scored.append((lbl, score))

    return sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]


In [38]:
def fast_sbert_fallback(title, lbl2ent_type, sbert_model):
    """
    NEW: Pure SBERT fallback for movie title resolution (no graph scans).
    """
    title = title.strip()
    candidates = []
    for lbl, entlist in lbl2ent_type.items():
        for label_type, uri in entlist:
            if any(k in label_type.lower() for k in ["film", "movie", "series", "episode"]):
                candidates.append((lbl, uri))

    if not candidates:
        return None

    cand_labels = [lbl for lbl, _ in candidates]
    cand_embs = sbert_model.encode(cand_labels, normalize_embeddings=True)
    q_emb = sbert_model.encode(title, normalize_embeddings=True)

    sims = util.cos_sim(q_emb, cand_embs)[0].cpu().numpy()
    idx = int(np.argmax(sims))
    return candidates[idx][1]   # URI


# GIZEMIN YAPTIĞI YERLER

In [39]:
# ==========================================================
# 🎬  Movie Recommendation Logic with Detailed Debug Prints
# ==========================================================

from curses import meta


def _extract_year(date_literal):
    """Extracts the year (int) from a date literal or string."""
    if isinstance(date_literal, Literal):
        val = str(date_literal)
    else:
        val = str(date_literal)
    m = re.search(r"(\d{4})", val)
    return int(m.group(1)) if m else None

def find_genre_predicates():
    """
    Use the relation extraction model to identify all predicates
    semantically close to 'genre' or 'type of movie'.
    Returns a list of (pid, label) tuples.
    """
    candidates = get_relation_candidates("genre of movie")  # reuse your model
    top_labels = [r["label"] for r in candidates if "genre" in r["label"].lower() or "type" in r["label"].lower()]
    if not top_labels:
        # fallback: just 'genre'
        top_labels = ["genre"]
    return top_labels


def get_movie_metadata(entity_uri, graph):
    """
    Collect metadata for a given movie from the graph.
    Returns a dictionary with genre, production company, year, 'based on', and sequel info.
    """
    meta = {
        "genres": set(),
        "prod_cos": set(),
        "years": set(),
        "based_on": set(),
        "types": set(),
        "has_sequel": False
    }

    P_GENRE = URIRef("http://www.wikidata.org/prop/direct/P136")
    P_RELEASE = URIRef("http://www.wikidata.org/prop/direct/P577")
    P_PROD_CO = URIRef("http://www.wikidata.org/prop/direct/P272")
    P_BASED_ON = URIRef("http://www.wikidata.org/prop/direct/P144")
    P_FOLLOWED_BY = URIRef("http://www.wikidata.org/prop/direct/P156")
    P_INSTANCE_OF = URIRef("http://www.wikidata.org/prop/direct/P31")

    # ----- GENRE -----
        # ----- GENRE (via relation extraction logic) -----
    genre_labels = find_genre_predicates()  # get semantically similar predicates
    matched_any = False

    for genre_label in genre_labels:
        pid, label = find_exact_predicate(genre_label)
        if not pid:
            pid, label, score = _hybrid_best_predicate(genre_label)

        if pid:
            pred_uri = URIRef(f"http://www.wikidata.org/prop/direct/{pid.upper()}")
            for _, _, obj in graph.triples((entity_uri, pred_uri, None)):
                lbl = ent2lbl.get(obj)
                if lbl:
                    meta["genres"].add(lbl)
                    matched_any = True

    # Fallback if nothing found (default P136)
    if not matched_any:
        P_GENRE = URIRef("http://www.wikidata.org/prop/direct/P136")
        for _, _, obj in graph.triples((entity_uri, P_GENRE, None)):
            lbl = ent2lbl.get(obj)
            if lbl:
                meta["genres"].add(lbl)


    # ----- PROD COMPANY -----
    for _, _, obj in graph.triples((entity_uri, P_PROD_CO, None)):
        lbl = ent2lbl.get(obj)
        if lbl:
            meta["prod_cos"].add(lbl)

    # ----- RELEASE YEAR -----
    # Try direct triple lookup (URI-based)
    found_release = False
    for _, _, obj in graph.triples((entity_uri, P_RELEASE, None)):
        year = _extract_year(obj)
        if year:
            decade = (year // 10) * 10
            meta["years"].add(decade)
            found_release = True

    # Fallback: search by label literal if nothing found
    if not found_release:
        label = ent2lbl.get(entity_uri)
        if label:
            for subj, _, obj in graph.triples((None, P_RELEASE, None)):
                # check if subject has same label literal
                for _, _, subj_label in graph.triples((subj, RDFS.label, None)):
                    if str(subj_label).lower() == label.lower():
                        year = _extract_year(obj)
                        if year:
                            decade = (year // 10) * 10
                            meta["years"].add(decade)


    # ----- CLASSIC DETECTION (without using 'based on' relation) -----
    classics_keywords = {"shakespeare", "dickens", "austen", "tolstoy", "brontë",
                        "homer", "goethe", "literary", "novel", "play", "tragedy"}

    is_classic = False

    # Author of the movie (if any)
    for _, _, author in graph.triples((entity_uri, URIRef("http://www.wikidata.org/prop/direct/P50"), None)):
        author_lbl = ent2lbl.get(author, "").lower()
        if any(k in author_lbl for k in classics_keywords):
            is_classic = True

    # Instance of (type of the movie itself)
    for _, _, inst in graph.triples((entity_uri, URIRef("http://www.wikidata.org/prop/direct/P31"), None)):
        inst_lbl = ent2lbl.get(inst, "").lower()
        if any(k in inst_lbl for k in classics_keywords):
            is_classic = True

    # after a work by
    for _, _, inst in graph.triples((entity_uri, URIRef("http://www.wikidata.org/prop/direct/P1877"), None)):
        inst_lbl = ent2lbl.get(inst, "").lower()
        if any(k in inst_lbl for k in classics_keywords):
            is_classic = True

    if is_classic:
        meta["based_on"].add("classic_literature")


    # ----- SEQUEL -----
    for _, _, obj in graph.triples((entity_uri, P_FOLLOWED_BY, None)):
        meta["has_sequel"] = True

    # ----- INSTANCE TYPES -----
    types = []
    for _, _, obj in graph.triples((entity_uri, P_INSTANCE_OF, None)):
        lbl = ent2lbl.get(obj)
        if lbl:
            types.append(lbl)
            meta["types"].add(lbl)

    # Debug summary
    print(f"  DEBUG: Entity {ent2lbl.get(entity_uri, entity_uri.split('/')[-1])}")
    print(f"    Found {len(meta['genres'])} genres, {len(meta['years'])} release values, "
          f"{len(meta['prod_cos'])} prod companies, {len(types)} types")
    if meta["genres"]:
        print(f"    Genres found: {list(meta['genres'])[:3]}{'...' if len(meta['genres'])>3 else ''}")
    if meta["based_on"]:
        print(f"    Based on classics detected.")
    if meta["prod_cos"]:
        print(f"    Prod companies found: {list(meta['prod_cos'])[:3]}{'...' if len(meta['prod_cos'])>3 else ''}")
    if meta["years"]:
        print(f"    Release years found: {list(meta['years'])[:3]}{'...' if len(meta['years'])>3 else ''}")
    if types:
        print(f"    Types found: {types[:3]}{'...' if len(types)>3 else ''}")

    return meta


def _most_common(items):
    if not items:
        return []
    from collections import Counter
    cnt = Counter(items)
    max_freq = max(cnt.values())
    
    # ✅ If there is only one unique item, return it
    if len(cnt) == 1:
        return list(cnt.keys())
    
    # ✅ If multiple items have the same frequency, return all
    most_common = [x for x, c in cnt.items() if c == max_freq]
    return most_common



def _summarize_years(years):
    if not years:
        return ""
    years = sorted(list(set(years)))
    if len(years) == 1:
        return str(f"{years[0]}s")
    common_years = _most_common(years)
    if not common_years:
        return ""
    common_years = sorted(common_years)
    if len(common_years) == 1:
        return str(f"{common_years[0]}s")
    decade_start = common_years[0]
    decade_end = common_years[-1]
    return f"{decade_start}s" if decade_start == decade_end else f"{decade_start}s–{decade_end}s"

metadata_cache = {}

def summarize_preferences(movie_uris, graph):
    print("\nBuilding profile from movies...")

    combined = {
        "genres": [],
        "prod_cos": [],
        "years": [],
        "based_on": [],
        "types": [],
        "has_sequel": 0
    }

    for uri in movie_uris:

        # --- FAST CACHE ---
        if uri not in metadata_cache:
            metadata_cache[uri] = get_movie_metadata(uri, graph)
        meta = metadata_cache[uri]
        # -------------------

        for k in combined:
            if k == "has_sequel":
                combined[k] += int(meta["has_sequel"])
            else:
                combined[k].extend(list(meta[k]))

    # keep only strong signals (your logic unchanged)
    def dominant_only(items):
        from collections import Counter
        cnt = Counter(items)
        if not cnt:
            return []
        max_freq = max(cnt.values())
        if max_freq < 2:
            return []
        return [x for x, c in cnt.items() if c == max_freq]

    dominant_genres  = dominant_only(combined["genres"])
    dominant_prodcos = dominant_only(combined["prod_cos"])
    dominant_years   = dominant_only(combined["years"])
    classic_source   = len(combined["based_on"]) > 0
    sequel_interest  = combined["has_sequel"] > 0
    dominant_types   = dominant_only(combined["types"])

    summary = {
        "genres": dominant_genres,
        "years": _summarize_years(dominant_years),
        "prod_cos": dominant_prodcos,
        "classic": classic_source,
        "sequels": sequel_interest,
        "types": dominant_types
    }

    print("\n--- SUMMARY ---")
    print(summary)
    print("------------------\n")
    return summary




def generate_recommendation(summary):
    rec_parts = []

    if summary["genres"]:
        rec_parts.append(f"{'s and '.join(summary['genres'])}s")
    if summary["years"]:
        rec_parts.append(f"from the {summary['years']}")
    if summary["classic"]:
        rec_parts.append("based on classic literature")
    if summary["prod_cos"]:
        rec_parts.append(f"produced by {' or '.join(summary['prod_cos'])} companies")
    if summary["sequels"]:
        rec_parts.append("or sequels to those movies")
    if summary["types"]:
        clean_types = [t for t in summary["types"] if t.lower().strip() != "film"]
        if clean_types:
            rec_parts.append(f"of type {' and '.join(clean_types)}")


    if not rec_parts:
        return "I think this movie preference profile is quite unique, so I don't have specific recommendations."

    sentence = "Adequate recommendations would be " + ", ".join(rec_parts) + "."
    return sentence


def recommend_from_liked_movies(movie_labels, graph=g):
    """
    Given a list of liked movie names, analyze them and generate a recommendation sentence.
    Includes detailed debug prints.
    """
    print(f"\nDetected recommendation question:")
    print(f"  Given that I like {', '.join(repr(m) for m in movie_labels)}, can you recommend some movies?")
    print("\nExtracting titles...")

    movie_uris = []
    for title in movie_labels:
        print(f"  Looking for movie: '{title}'")
        uri = find_movie_uri_by_label(ent2lbl, title, g)
        if uri:
            short_qid = uri.split('/')[-1]
            lbl = ent2lbl.get(uri, title)
            print(f"    Found exact match: {lbl} ({short_qid})")
            movie_uris.append(uri)
        else:
            print(f"    ❌ Could not find match for '{title}'")

    print(f"\nFound {len(movie_uris)}/{len(movie_labels)} movies in knowledge graph\n")

    if not movie_uris:
        return "I couldn't find any of those movies in the knowledge graph."

    summary = summarize_preferences(movie_uris, graph)
    recommendation = generate_recommendation(summary)
    print(f"✅ Final Recommendation: {recommendation}\n")
    return recommendation


In [40]:
def enrich_recommendation_sbert(movie_uris, graph, ent2lbl, sbert_model, summary, top_k=3):
    """
    Parallel enrichment: uses ONLY the metadata summary (genres, types, production company...)
    and SBERT similarity — NO graph traversal.
    Does not change existing logic.
    """

    labels = []

    labels.extend(summary.get("genres", []))
    labels.extend(summary.get("types", []))
    labels.extend(summary.get("prod_cos", []))

    if summary.get("years"):
        labels.append(summary["years"])

    if summary.get("classic"):
        labels.append("classic literature")

    if summary.get("sequels"):
        labels.append("movie sequels")

    labels = list({lbl for lbl in labels if lbl})
    if not labels:
        return ""

    print(f"⚙️ SBERT enrichment on {len(labels)} labels...")

    # SBERT encode small local set
    emb = {lbl: sbert_model.encode(lbl, normalize_embeddings=True) for lbl in labels}

    sims = []
    from itertools import combinations
    for a, b in combinations(labels, 2):
        score = float(util.cos_sim(emb[a], emb[b]))
        sims.append((a, b, score))

    sims = sorted(sims, key=lambda x: x[2], reverse=True)
    if not sims:
        return ""

    top_pairs = sims[:min(len(sims), 6)]
    similar_labels = list({lbl for a, b, _ in top_pairs for lbl in (a, b)})

    parts = []

    if similar_labels:
        parts.append(
            "you seem to enjoy themes related to "
            + ", ".join(similar_labels[:3])
        )

    a, b, score = top_pairs[0]
    parts.append(f"for example, '{a}' and '{b}' are strongly related in your preferences")

    sentence = "Additionally, " + " ".join(parts) + "."
    return sentence


In [41]:
meaningful_relations = {
    "P136", "P179", "P272", "P155", "P156", "P57", "P58",
    "P161", "P1877", "P364", "P361"
}

def enrich_recommendation_from_similarity(
    movie_uris, graph, ent2lbl, sbert_model,
    per_movie_limit=5, top_k=3
):
    """
    Very fast enrichment using SBERT only.
    - Extracts up to N related entities via meaningful relations.
    - Computes semantic similarity among them.
    - Suggests related label + actual movie.
    """
    from collections import defaultdict
    related = defaultdict(set)

    # --- 1) Collect meaningful labels ---
    for uri in movie_uris:
        mref = URIRef(uri)
        count = 0
        for _, rel, obj in graph.triples((mref, None, None)):
            pid = rel.split("/")[-1].upper()
            if pid in meaningful_relations and isinstance(obj, URIRef):
                lbl = ent2lbl.get(obj)
                if lbl:
                    related[uri].add(lbl)
                    count += 1
                    if count >= per_movie_limit:
                        break

    all_labels = list({lbl for s in related.values() for lbl in s})
    if not all_labels:
        return ""

    # --- 2) SBERT encode all labels ---
    emb = sbert_model.encode(all_labels, normalize_embeddings=True)

    # --- 3) Compute label-to-label similarities ---
    sims = util.cos_sim(emb, emb).cpu().numpy()

    # top similar pairs
    pairs = []
    for i in range(len(all_labels)):
        for j in range(i+1, len(all_labels)):
            if sims[i, j] > 0.35:
                pairs.append((all_labels[i], all_labels[j], sims[i, j]))

    pairs = sorted(pairs, key=lambda x: x[2], reverse=True)
    if not pairs:
        return ""

    top_labels = list({l for p in pairs[:10] for l in p})

    # --- 4) Try to find actual movies linked to these labels ---
    candidates = set()
    for lbl in top_labels:
        for subj, _, obj in graph.triples((None, None, None)):
            if ent2lbl.get(obj) == lbl:
                sl = ent2lbl.get(subj)
                if sl and sl not in [ent2lbl.get(URIRef(u)) for u in movie_uris]:
                    candidates.add(sl)
                    if len(candidates) >= top_k:
                        break
        if len(candidates) >= top_k:
            break

    # --- 5) Build sentence ---
    parts = []
    if top_labels:
        parts.append(f"you seem to enjoy films related to {', '.join(top_labels[:3])}")
    if candidates:
        parts.append(f"An example similar movie could be {', '.join(list(candidates)[:2])}")

    if not parts:
        return ""

    return "Additionally, " + " ".join(parts) + "."


In [42]:
import random
import numpy as np

# ============================
# FAST RANDOM-WALK 2-HOP SEARCH
# ============================


In [43]:

# ======================================================
# NEW, FIXED, RELIABLE MOVIE DETECTOR
# ======================================================
def _is_movie_uri(uri, graph, ent2lbl):
    """
    Accepts movies, series, episodes, animations.
    Rejects obvious non-content entities.
    Only requires one P31 = instance-of.
    """
    lbl = ent2lbl.get(uri, "").lower()
    if not lbl:
        return False

    banned_terms = [
        "genre", "category", "list of",
        "studio", "company", "franchise",
        "soundtrack", "character", "award"
    ]
    if any(term in lbl for term in banned_terms):
        return False

    P31 = URIRef("http://www.wikidata.org/prop/direct/P31")
    for _, _, obj in graph.triples((uri, P31, None)):
        return True

    return False



# ======================================================
# HELPER: Collect all metadata signals from Part B
# ======================================================
def extract_rw_signals(metadata_summary):
    """
    Returns dict with movie metadata signals for guiding the walk.
    """
    signals = {
        "genres": set(metadata_summary.get("genres", [])),
        "types": set(metadata_summary.get("types", [])),
        "prod_cos": set(metadata_summary.get("prod_cos", [])),
        "years": metadata_summary.get("years", "")
    }
    return signals



# ======================================================
# NEW, MULTI-SIGNAL RANDOM WALK EXPLORER
# ======================================================
def biased_randomwalk_candidates(movie_uris, graph, ent2lbl, metadata_signals,
                                 max_neighbors=50):
    """
    Multi-signal random walk:
       - GENRE (P136)
       - TYPE  (P31)
       - COMPANY (P272)
       - DECADE  (P577 year extraction)
    Each produces candidate movies with weights.
    """

    P_GENRE   = URIRef("http://www.wikidata.org/prop/direct/P136")
    P_TYPE    = URIRef("http://www.wikidata.org/prop/direct/P31")
    P_COMPANY = URIRef("http://www.wikidata.org/prop/direct/P272")
    P_RELEASE = URIRef("http://www.wikidata.org/prop/direct/P577")

    candidates = {}   # uri → score

    # ----------------------------------------------------
    # Extract usable decade
    # ----------------------------------------------------
    decade_target = None
    if metadata_signals["years"]:
        try:
            decade_target = int(metadata_signals["years"][:4])
        except:
            decade_target = None

    # ----------------------------------------------------
    # WALK FOR EACH MOVIE
    # ----------------------------------------------------
    for m in movie_uris:
        mref = URIRef(m)

        # =================================================
        # HOP 1: via GENRE
        # =================================================
        for _, _, genre_node in graph.triples((mref, P_GENRE, None)):

            g_lbl = ent2lbl.get(genre_node, "").lower()
            if g_lbl in metadata_signals["genres"]:

                # Hop 2: genre_node → movies
                count = 0
                for subj, rel, obj in graph.triples((None, None, genre_node)):
                    if count >= max_neighbors: break
                    count += 1

                    if not _is_movie_uri(subj, graph, ent2lbl):
                        continue

                    candidates[subj] = candidates.get(subj, 0) + 3


        # =================================================
        # HOP 1: via TYPE
        # =================================================
        for _, _, type_node in graph.triples((mref, P_TYPE, None)):

            t_lbl = ent2lbl.get(type_node, "").lower()
            if t_lbl in metadata_signals["types"]:

                # Hop 2: type_node → movies
                count = 0
                for subj, rel, obj in graph.triples((None, None, type_node)):
                    if count >= max_neighbors: break
                    count += 1

                    if not _is_movie_uri(subj, graph, ent2lbl):
                        continue

                    candidates[subj] = candidates.get(subj, 0) + 3


        # =================================================
        # HOP 1: via PRODUCTION COMPANY
        # =================================================
        for _, _, co_node in graph.triples((mref, P_COMPANY, None)):

            co_lbl = ent2lbl.get(co_node, "").lower()
            if co_lbl in metadata_signals["prod_cos"]:

                count = 0
                for subj, rel, obj in graph.triples((None, None, co_node)):
                    if count >= max_neighbors: break
                    count += 1

                    if not _is_movie_uri(subj, graph, ent2lbl):
                        continue

                    candidates[subj] = candidates.get(subj, 0) + 2


        # =================================================
        # RELEASE DECADE MATCHING
        # =================================================
        if decade_target is not None:
            for subj, rel, obj in graph.triples((None, P_RELEASE, None)):

                year = _extract_year(obj)
                if not year: 
                    continue

                decade = (year // 10) * 10
                if decade == decade_target:

                    if not _is_movie_uri(subj, graph, ent2lbl):
                        continue

                    candidates[subj] = candidates.get(subj, 0) + 1


    # ---------------------------------------------------------
    # Convert back to (label, uri) pairs
    # ---------------------------------------------------------
    results = []
    for uri, weight in candidates.items():
        lbl = ent2lbl.get(uri)
        if lbl:
            results.append((lbl, uri, weight))

    # sort by weight
    results = sorted(results, key=lambda x: x[2], reverse=True)

    # return WITHOUT weight — scoring step added later
    return [(lbl, uri) for (lbl, uri, _) in results[:max_neighbors]]





def score_randomwalk_movies(liked_labels, candidate_pairs, sbert_model,
                            top_k=5, min_score=0.45):
    if not candidate_pairs:
        return []

    cand_labels = [lbl for lbl, _ in candidate_pairs]
    liked_embs = sbert_model.encode(liked_labels, normalize_embeddings=True)
    cand_embs  = sbert_model.encode(cand_labels, normalize_embeddings=True)

    scored = []
    for i, (lbl, uri) in enumerate(candidate_pairs):
        sims = np.dot(liked_embs, cand_embs[i])
        score = float(np.mean(sims))
        if score >= min_score:
            scored.append((lbl, uri, score))

    return sorted(scored, key=lambda x: x[2], reverse=True)[:top_k]



def randomwalk_to_sentence(scored):
    if not scored:
        return "Additionally, I could not find any movie-based suggestions."

    labels = [lbl for lbl, _, _ in scored]

    if len(labels) == 1:
        return f"Additionally, you may also enjoy {labels[0]}."
    if len(labels) == 2:
        return f"Additionally, you may also enjoy {labels[0]} or {labels[1]}."
    return f"Additionally, you may also enjoy {labels[0]}, {labels[1]}, or {labels[2]}."



In [44]:
def label_to_movie_uri(lbl, lbl2ent_type):
    """
    NEW: Fast label → URI mapping using movie-like type labels.
    """
    if lbl not in lbl2ent_type:
        return None
    for type_label, uri in lbl2ent_type[lbl]:
        if any(k in type_label.lower() for k in ["film", "movie", "series", "episode"]):
            return uri
    return None


In [45]:
def recommend_fast_from_movies(movie_uris, graph, ent2lbl, lbl2ent_type, sbert_model, top_k=5):
    """
    NEW: Unified fast recommendation engine.
    """
    # --- Step 1: 1-hop metadata per movie
    meta_list = [
        get_fast_onehop_metadata(uri, graph, ent2lbl, meaningful_relations)
        for uri in movie_uris
    ]

    # --- Step 2: centroids
    movie_centroids = [
        compute_movie_centroid(meta, sbert_model)
        for meta in meta_list
        if compute_movie_centroid(meta, sbert_model) is not None
    ]

    # --- Step 3: merged candidate space
    merged_labels = merge_metadata_list(meta_list)

    # --- Step 4: score candidates
    ranked = rank_metadata_candidates(
        merged_labels, movie_centroids, sbert_model, top_k=top_k
    )

    # --- Step 5: map to movie URIs
    final_movies = []
    for lbl, score in ranked:
        uri = label_to_movie_uri(lbl, lbl2ent_type)
        if uri:
            final_movies.append((lbl, uri, score))

    return final_movies[:top_k]


In [46]:
def get_rid_of(message: str) -> str:
    """
    Removes boilerplate phrases such as:
    - 'Please answer this question with a factual/embedding approach:'
    - 'Given that I like ...'
    - 'Recommend movies like ...'
    - 'Can you recommend ...'
    - 'Recommend ...'
    - 'I like ... can you recommend ...'

    Leaves movie titles intact.
    """

    msg = message.strip()

    # -------------------------------
    # 1) OLD EXISTING PATTERN (KEEP)
    # -------------------------------
    pattern_old = r"(please\s+answer\s+this\s+question\s+with\s+(a\s+)?(factual|embedding)\s+approach:?)"
    cleaned = re.sub(pattern_old, "", msg, flags=re.IGNORECASE)
    cleaned = re.sub(pattern_old + r"$", "", cleaned, flags=re.IGNORECASE)

    # -----------------------------------------------
    # 2) NEW RECOMMENDATION-BOILERPLATE REMOVAL (ADD)
    # -----------------------------------------------
    # These phrases wrap the movie titles but are not part of them
    rec_phrases = [
        r"given\s+that\s+i\s+like",
        r"i\s+like",
        r"can\s+you\s+recommend(\s+some\s+movies)?",
        r"can\s+you\s+recommend",
        r"recommend\s+movies\s+like",
        r"recommend\s+movies",
        r"recommend\s+some\s+movies",
        r"recommend",
        r"movies\s+like"
    ]

    for ph in rec_phrases:
        cleaned = re.sub(ph, "", cleaned, flags=re.IGNORECASE)

    # Strip leading "and" or commas if left behind
    cleaned = cleaned.strip(" ,:;.-").strip()

    # Capitalize first letter if needed
    if cleaned and cleaned[0].islower():
        cleaned = cleaned[0].upper() + cleaned[1:]

    return cleaned


In [47]:
def recommend(question_text):
    """
    FINAL BOT ENTRYPOINT.
    Takes a natural-language movie preference question,
    extracts movie names, resolves them to URIs,
    and returns recommended movies.
    """

    print("\n[Recommender] Raw question:", question_text)

    # ---------------------------------------------------------
    # 1) First clean the question using your updated get_rid_of
    # ---------------------------------------------------------
    cleaned = get_rid_of(question_text)
    print("[Recommender] Cleaned question:", cleaned)

    if not cleaned:
        return ["I couldn't detect any movie titles."]

    # ---------------------------------------------------------
    # 2) Extract entities (movies) from cleaned question
    # ---------------------------------------------------------
    parsed = extract_relations_and_entities(cleaned)
    entities = parsed.get("entity_candidates", [])

    movie_names = [e["text"].strip() for e in entities if len(e["text"].strip()) > 1]

    print("[Recommender] Detected movie names:", movie_names)

    if not movie_names:
        return ["I couldn't detect any movie titles."]

    # ---------------------------------------------------------
    # 3) Resolve movie names to URIs
    # ---------------------------------------------------------
    movie_uris = []

    for name in movie_names:
        # First try exact match
        uri = find_uri_by_label(ent2lbl, name)
        if uri:
            movie_uris.append(uri)
            continue

        # Fallback SBERT mapping
        fallback_uri = fast_sbert_fallback(name, lbl2ent_type, sbert_model)
        if fallback_uri:
            movie_uris.append(fallback_uri)

    if not movie_uris:
        return ["I could not resolve any of these movies in the KG."]

    print("[Recommender] Resolved URIs:", movie_uris)

    # ---------------------------------------------------------
    # 4) Run the fast recommendation engine
    # ---------------------------------------------------------
    recs = recommend_fast_from_movies(
        movie_uris=movie_uris,
        graph=g,
        ent2lbl=ent2lbl,
        lbl2ent_type=lbl2ent_type,
        sbert_model=sbert_model,
        top_k=5
    )

    if not recs:
        return ["No recommendations found."]

    # ---------------------------------------------------------
    # 5) Format output (bot-friendly)
    # ---------------------------------------------------------
    formatted = [
        f"{lbl} (score={score:.3f})"
        for (lbl, uri, score) in recs
    ]

    return formatted


In [48]:
recommend("Given that I like The Lion King, 'Pocahontas', and 'The Beauty and the Beast', can you recommend some movies?")



[Recommender] Raw question: Given that I like The Lion King, 'Pocahontas', and 'The Beauty and the Beast', can you recommend some movies?
[Recommender] Cleaned question: The Lion King, 'Pocahontas', and 'The Beauty and the Beast', ?

🔍 Entity 'The Lion King' disambiguation:
   Option 1: media franchise → the Disney The Lion King, 'Pocahontas', and 'The Beauty and the Beast', ? (score=0.945)
   Option 2: dramatico-musical work → the musical The Lion King, 'Pocahontas', and 'The Beauty and the Beast', ? (score=0.889)
   Option 3: animated film → the film The Lion King, 'Pocahontas', and 'The Beauty and the Beast', ? (score=0.853)
   Option 4: film → the film The Lion King, 'Pocahontas', and 'The Beauty and the Beast', ? (score=0.853)
   Option 5: video game → the video game The Lion King, 'Pocahontas', and 'The Beauty and the Beast', ? (score=0.838)
   → Selected: the Disney The Lion King, 'Pocahontas', and 'The Beauty and the Beast', ? [Q196600]

🔍 Entity 'Pocahontas' disambiguation:
 

['No recommendations found.']

# Lets Try Something New

In [49]:
movies  = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
tags    = pd.read_csv("tags.csv")
links   = pd.read_csv("links.csv")

In [50]:
print(movies.head())
print(ratings.head())


   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1       17     4.0  944249077
1       1       25     1.0  944250228
2       1       29     2.0  943230976
3       1       30     5.0  944249077
4       1       32     5.0  943228858


In [51]:
def normalize_title(t):
    return t.lower().strip().replace("(", "").replace(")", "")

movies["norm_title"] = movies["title"].apply(normalize_title)
movies["genres_list"] = movies["genres"].apply(lambda x: x.split("|") if isinstance(x, str) else [])


In [52]:
model_sbert = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")




In [53]:
print("Loading cached MovieLens embeddings…")

movie_embeddings = torch.load("movielens_embeddings.pt")
movies = pd.read_csv("movielens_movies_cached.csv")

print("Loaded:", movie_embeddings.shape)



Loading cached MovieLens embeddings…
Loaded: torch.Size([87585, 768])


In [54]:
def find_movielens_match(user_title):
    q = normalize_title(user_title)
    q_emb = model_sbert.encode(q, convert_to_tensor=True, normalize_embeddings=True)

    scores = util.cos_sim(q_emb, movie_embeddings)[0]
    best_idx = int(scores.argmax())
    best_score = float(scores[best_idx])

    if best_score < 0.6:
        return None  # too weak

    return movies.iloc[best_idx]


In [55]:
rating_stats = ratings.groupby("movieId")["rating"].agg(["mean", "count"])
rating_stats["weighted"] = rating_stats["mean"] * np.log1p(rating_stats["count"])


In [56]:
def get_genre_candidates(liked_movies, movies_df):
    liked_genres = set()
    for mv in liked_movies:
        liked_genres.update(mv["genres_list"])

    # remove "(no genres listed)" if present
    liked_genres.discard("(no genres listed)")

    # return movies sharing at least 1 genre
    mask = movies_df["genres_list"].apply(lambda gs: len(set(gs) & liked_genres) > 0)
    return movies_df[mask]


In [57]:
def score_candidates(liked_movies, candidates_df):
    # 1) Build liked genre set
    liked_genres = set()
    for mv in liked_movies:
        liked_genres.update(mv["genres_list"])

    # 2) Compute overlaps (vectorized)
    overlaps = (
        candidates_df["genres_list"]
        .apply(lambda gs: len(set(gs) & liked_genres))
        .values
    )

    # Keep only movies with overlap ≥ 1
    mask = overlaps > 0
    cand = candidates_df[mask]
    overlaps = overlaps[mask]

    # 3) SAFE RATING LOOKUP (no errors)
    movieIds = cand["movieId"].values

    # Use reindex → missing entries = NaN → fill with 0
    rating_vec = (
        rating_stats.reindex(movieIds)["weighted"]
        .fillna(0)
        .values
    )

    # 4) Vectorized scoring
    final_scores = overlaps * 4.2 + rating_vec

    # 5) Select top 10
    idx = np.argsort(-final_scores)[:10]

    titles = cand["title"].values[idx]
    scores = final_scores[idx]

    return list(zip(titles, scores))


In [58]:
def movielens_genre_profile(matched_movies):
    """
    matched_movies = list of MovieLens rows returned by find_movielens_match().
    Each contains a 'genres_list' column already parsed.
    """
    genre_count = {}

    for mv in matched_movies:
        for g in mv["genres_list"]:
            if g.lower() == "no genres listed":
                continue
            genre_count[g] = genre_count.get(g, 0) + 1

    if not genre_count:
        return []

    # sort by frequency
    ranked = sorted(genre_count.items(), key=lambda x: x[1], reverse=True)
    return [g for g, c in ranked[:3]]  # top 3 genres


In [59]:
def recommend_movielens(user_titles):
    matched = []
    for t in user_titles:
        m = find_movielens_match(t)
        if m is not None:
            matched.append(m)

    if not matched:
        return [], [], []     # (matched_movies, candidates, scored)

    candidates = get_genre_candidates(matched, movies)
    scored = score_candidates(matched, candidates)

    return matched, candidates, scored


# Something Very New - Using the given data.  

# =========================================================
# 1. Strict Data Loading (No Backups)
# =========================================================

In [60]:


print("Loading data from CSVs...")

# Direct loading. This will raise FileNotFoundError if files are missing.
reviews_df = pd.read_csv("user_comments.csv")
plots_df = pd.read_csv("movie_plots.csv")

print(f"Loaded {len(reviews_df)} reviews and {len(plots_df)} plots.")




Loading data from CSVs...
Loaded 46186 reviews and 12376 plots.


# =========================================================
# 2. Generate Movie Embeddings (The Space Basis)
# =========================================================

In [61]:
plots_df.head()

,index,plot
0,http://www.wikidata.org/entity/Q612373,"In the future, the inmates of a private underg..."
1,http://www.wikidata.org/entity/Q2089577,"A bashful bachelor penguin named Hubie, who's ..."
2,http://www.wikidata.org/entity/Q5227053,"In 1026, students in western China are shangha..."
3,http://www.wikidata.org/entity/Q9077202,Angela and her young son Guille travel to the ...
4,http://www.wikidata.org/entity/Q844679,A wild and rollicking martial arts fantasy ext...


In [62]:
reviews_df.head()

,index,rating,sentiment,comment
0,http://www.wikidata.org/entity/Q2054210,4,0,I rented I AM CURIOUS-YELLOW from my video sto...
1,http://www.wikidata.org/entity/Q2054210,1,0,"""I Am Curious: Yellow"" is a risible and preten..."
2,http://www.wikidata.org/entity/Q2054210,1,0,If only to avoid making this type of film in t...
3,http://www.wikidata.org/entity/Q2054210,2,0,This film was probably inspired by Godard's Ma...
4,http://www.wikidata.org/entity/Q2054210,1,0,"Oh, brother...after hearing about this ridicul..."


In [63]:
"""print("Generating combined movie embeddings (plots + reviews + metadata)...")

from sentence_transformers import SentenceTransformer
import numpy as np
from collections import defaultdict

# ============= 1. Load Encoder =============
model = SentenceTransformer("all-MiniLM-L6-v2")

# ============= 2. Plot embeddings (384-d) =============
plot_emb = model.encode(
    plots_df["plot"].tolist(),
    show_progress_bar=True
)
movie_ids = plots_df["index"].tolist()
plot_emb_dict = dict(zip(movie_ids, plot_emb))

# ============= 3. Review embeddings (text 384-d) =============
review_text_emb = model.encode(
    reviews_df["comment"].astype(str).tolist(),
    show_progress_bar=True
)

# ============= 4. Metadata emb (2-d) =============
rating_scaled = (reviews_df["rating"] - 5.5) / 4.5
sent_scaled   = reviews_df["sentiment"] * 2 - 1

meta_emb = np.vstack([rating_scaled, sent_scaled]).T   # (N, 2)

# ============= 5. Combined review emb (386-d) =============
combined_review_emb = np.hstack([review_text_emb, meta_emb])"""



'print("Generating combined movie embeddings (plots + reviews + metadata)...")\n\nfrom sentence_transformers import SentenceTransformer\nimport numpy as np\nfrom collections import defaultdict\n\n# ============= 1. Load Encoder =============\nmodel = SentenceTransformer("all-MiniLM-L6-v2")\n\n# ============= 2. Plot embeddings (384-d) =============\nplot_emb = model.encode(\n    plots_df["plot"].tolist(),\n    show_progress_bar=True\n)\nmovie_ids = plots_df["index"].tolist()\nplot_emb_dict = dict(zip(movie_ids, plot_emb))\n\n# ============= 3. Review embeddings (text 384-d) =============\nreview_text_emb = model.encode(\n    reviews_df["comment"].astype(str).tolist(),\n    show_progress_bar=True\n)\n\n# ============= 4. Metadata emb (2-d) =============\nrating_scaled = (reviews_df["rating"] - 5.5) / 4.5\nsent_scaled   = reviews_df["sentiment"] * 2 - 1\n\nmeta_emb = np.vstack([rating_scaled, sent_scaled]).T   # (N, 2)\n\n# ============= 5. Combined review emb (386-d) =============\ncomb

In [64]:
"""# ============= 6. Aggregate review embeddings per movie =============
movie_to_reviews = defaultdict(list)

# movie IDs are stored in the index
for mid, emb in zip(reviews_df.index, combined_review_emb):
    movie_to_reviews[mid].append(emb)

# mean aggregation
movie_review_emb = {
    mid: np.mean(vecs, axis=0)
    for mid, vecs in movie_to_reviews.items()
}

# ============= 7. Pad plot embeddings (384 → 386 dims) =============
plot_padded_dict = {
    mid: np.pad(plot_emb_dict[mid], (0, 2), mode="constant")
    for mid in plot_emb_dict
}

# ============= 8. Weighted fusion =============
combined_movie_embeddings = {}

for mid in plot_padded_dict:
    if mid in movie_review_emb:    # only movies with reviews
        plot_vec = plot_padded_dict[mid]
        rev_vec  = movie_review_emb[mid]
        combined = 0.6 * plot_vec + 0.4 * rev_vec
        combined_movie_embeddings[mid] = combined

"print(f"Created {len(combined_movie_embeddings)} combined embeddings.")"""

'# ============= 6. Aggregate review embeddings per movie =============\nmovie_to_reviews = defaultdict(list)\n\n# movie IDs are stored in the index\nfor mid, emb in zip(reviews_df.index, combined_review_emb):\n    movie_to_reviews[mid].append(emb)\n\n# mean aggregation\nmovie_review_emb = {\n    mid: np.mean(vecs, axis=0)\n    for mid, vecs in movie_to_reviews.items()\n}\n\n# ============= 7. Pad plot embeddings (384 → 386 dims) =============\nplot_padded_dict = {\n    mid: np.pad(plot_emb_dict[mid], (0, 2), mode="constant")\n    for mid in plot_emb_dict\n}\n\n# ============= 8. Weighted fusion =============\ncombined_movie_embeddings = {}\n\nfor mid in plot_padded_dict:\n    if mid in movie_review_emb:    # only movies with reviews\n        plot_vec = plot_padded_dict[mid]\n        rev_vec  = movie_review_emb[mid]\n        combined = 0.6 * plot_vec + 0.4 * rev_vec\n        combined_movie_embeddings[mid] = combined\n\n"print(f"Created {len(combined_movie_embeddings)} combined embeddin

# Save to disk

In [ ]:
"""import numpy as np
import json
from sklearn.neighbors import NearestNeighbors
import pickle
import matplotlib.pyplot as plt

# ============================================================
# 1. Convert dict → matrix + ID list
# ============================================================
movie_ids = list(combined_movie_embeddings.keys())
embedding_matrix = np.vstack([combined_movie_embeddings[mid] for mid in movie_ids])

print("Original embedding matrix shape:", embedding_matrix.shape)

# ============================================================
# 2. Normalize embeddings once (for cosine similarity)
# ============================================================
norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
embedding_matrix_norm = embedding_matrix / norms

# Save normalized embeddings
np.save("combined_movie_embeddings_norm.npy", embedding_matrix_norm)

# Save movie ID order
with open("movie_ids.json", "w") as f:
    json.dump(movie_ids, f)

print("✔ Saved normalized embeddings & movie IDs")"""


Original embedding matrix shape: (2038, 386)
✔ Saved normalized embeddings & movie IDs


In [ ]:
"""# ============================================================
# 3. Build NN index (cosine distance)
# ============================================================
nn = NearestNeighbors(metric="cosine", n_neighbors=30)
nn.fit(embedding_matrix_norm)

# Save NN Index
with open("nn_index.pkl", "wb") as f:
    pickle.dump(nn, f)

print("✔ Saved NN index")"""

✔ Saved NN index


# ============================================================
# LOADING PIPELINE
# ============================================================

In [65]:
import pickle

In [66]:

# Load normalized embeddings
embedding_matrix_norm = np.load("combined_movie_embeddings_norm.npy")

# Load movie IDs
with open("movie_ids.json") as f:
    movie_ids = json.load(f)

# Load NN index
with open("nn_index.pkl", "rb") as f:
    nn = pickle.load(f)

# Reconstruct lookup dictionary
combined_movie_embeddings_norm = {
    mid: emb for mid, emb in zip(movie_ids, embedding_matrix_norm)
}

print("Loaded everything successfully:", embedding_matrix_norm.shape)


Loaded everything successfully: (2038, 386)


/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator NearestNeighbors from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [67]:
lion_id = "http://www.wikidata.org/entity/Q36479"


In [68]:
def recommend_similar(movie_id, k=10):
    idx = movie_ids.index(movie_id)
    dist, inds = nn.kneighbors([embedding_matrix_norm[idx]], n_neighbors=k)
    return [movie_ids[i] for i in inds[0]]


In [69]:
print(f"Total movies in embedding space: {len(movie_ids)}")

# show first 10 IDs
print("\nSample movie IDs:")
for mid in movie_ids[:10]:
    print("  ", mid)


Total movies in embedding space: 2038

Sample movie IDs:
   http://www.wikidata.org/entity/Q2089577
   http://www.wikidata.org/entity/Q1866730
   http://www.wikidata.org/entity/Q153723
   http://www.wikidata.org/entity/Q1049417
   http://www.wikidata.org/entity/Q1084074
   http://www.wikidata.org/entity/Q5479580
   http://www.wikidata.org/entity/Q1509291
   http://www.wikidata.org/entity/Q260008
   http://www.wikidata.org/entity/Q2924761
   http://www.wikidata.org/entity/Q180695


In [70]:
def get_movie_info(movie_id):
    row = plots_df.loc[plots_df["index"] == movie_id]
    if len(row) == 0:
        return movie_id, "(no plot available)"
    plot = row["plot"].values[0]
    short = plot[:180].replace("\n", " ") + "..."
    return movie_id, short


In [71]:
def print_neighbors(movie_id, k=5):
    try:
        idx = movie_ids.index(movie_id)
    except ValueError:
        print("Movie not found in embedding space:", movie_id)
        return
    
    dist, inds = nn.kneighbors([embedding_matrix_norm[idx]], n_neighbors=k+1)
    neighbor_ids = [movie_ids[i] for i in inds[0]]

    print("\n=== TARGET MOVIE ===")
    mid, info = get_movie_info(movie_id)
    print(mid)
    print(info)

    print("\n=== NEIGHBORS ===")
    for nid in neighbor_ids[1:]:   # skip itself
        nm, np_info = get_movie_info(nid)
        print("\n→", nm)
        print("  ", np_info)


In [72]:
print_neighbors(movie_ids[5], k=5)



=== TARGET MOVIE ===
http://www.wikidata.org/entity/Q5479580
Episodic study of feelings where six stories intertwine over one Saturday in a small town. Two boys save a young woman, Anni, and her baby from suicide by drowning. Esa, a down-on-...

=== NEIGHBORS ===

→ http://www.wikidata.org/entity/Q1634355
   This film follows the daily lives of a group of people barely scraping by in a slum on the outskirts of Tokyo. Yet as desperate as their circumstances are, each of them—the homeles...

→ http://www.wikidata.org/entity/Q912049
   A musical romantic tragedy about a famous composer who moves back to his small hometown after having had heart troubles. His search for a simple everyday life leads him into teachi...

→ http://www.wikidata.org/entity/Q560006
   A young woman reunites with the father she's not seen since early childhood, also falls in love with a sailor who wishes to marry her, and eventually is forced to reveal to each ma...

→ http://www.wikidata.org/entity/Q1643166
   A

/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [73]:
def recommend_by_text(text, k=10):
    vec = model.encode([text])
    vec = vec / np.linalg.norm(vec)
    vec = np.pad(vec, ((0,0),(0,2)), mode='constant')  # match 386 dims
    dist, inds = nn.kneighbors(vec, n_neighbors=k)
    return [movie_ids[i] for i in inds[0]]


In [74]:
from rapidfuzz import fuzz, process

def detect_movie_id_from_text(text, movie_ids, plots_df, threshold=75):
    text_lower = text.lower().strip()

    # --- 1. Direct QID in text ---
    for mid in movie_ids:
        if mid.lower() in text_lower:
            return mid

    # --- 2. Fuzzy match QID strings ---
    matches = process.extract(text_lower, movie_ids, scorer=fuzz.partial_ratio, limit=3)
    if matches and matches[0][1] >= threshold:
        return matches[0][0]

    # --- 3. Fuzzy match against plot text ---
    best_mid = None
    best_score = threshold

    for mid, plot in zip(plots_df["index"], plots_df["plot"]):
        # ensure string
        if not isinstance(plot, str):
            continue
        
        score = fuzz.partial_ratio(text_lower, plot.lower())
        if score > best_score:
            best_mid = mid
            best_score = score

    return best_mid


In [75]:
def recommend_from_user_text(user_text, k=10):
    # Try to detect a specific movie reference
    detected_mid = detect_movie_id_from_text(user_text, movie_ids, plots_df)
    
    if detected_mid:
        print(f"Detected movie in text: {detected_mid}")

        # Use the movie's existing embedding
        plot_vec = combined_movie_embeddings_norm[detected_mid]
        query_vec = plot_vec.reshape(1, -1)

    else:
        print("No specific movie detected — using generic semantic embedding.")
        
        # Use SBERT sentence embedding
        sbert_vec = model.encode([user_text])
        sbert_vec = sbert_vec / np.linalg.norm(sbert_vec)
        sbert_vec = np.pad(sbert_vec, ((0,0),(0,2)), mode='constant')
        query_vec = sbert_vec

    # Query NN
    dist, inds = nn.kneighbors(query_vec, n_neighbors=k)
    return [movie_ids[i] for i in inds[0]]


In [76]:
def pretty_recommend_from_text(user_text, k=10):
    results = recommend_from_user_text(user_text, k)
    print("\n=== Input ===")
    print(user_text)

    print("\n=== Recommendations ===")
    for mid in results:
        mid2, info = get_movie_info(mid)
        print("\n→", mid2)
        print("  ", info)


In [77]:
pretty_recommend_from_text("I like Inception movie")


No specific movie detected — using generic semantic embedding.

=== Input ===
I like Inception movie

=== Recommendations ===

→ http://www.wikidata.org/entity/Q272608
   Blonde Betty Elms has only just arrived in Hollywood to become a movie star when she meets an enigmatic brunette with amnesia. Meanwhile, as the two set off to solve the second wom...

→ http://www.wikidata.org/entity/Q2765056
   A Canadian documentary featuring two young filmmakers attending the Toronto Film Festival to pitch a film concept to various celebrities. Their film idea, titled "The Dawn", concer...

→ http://www.wikidata.org/entity/Q2054210
   Lena, aged twenty, wants to know all she can about life and reality. She collects information on everyone and everything, storing her findings in an enormous archive. She experimen...

→ http://www.wikidata.org/entity/Q3223660
   Deep in the desert, a rural American family is forced to endure a night of inter-dimensional, extra terrestrial terror when a UFO appears o

/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Combination

In [78]:
import time

In [ ]:
def recommend_combined(question_text):

    t0 = time.perf_counter()
    print("\n[Recommender] Raw question:", question_text)

    # ----------------------------------------------------------
    # 1) Cleaning
    # ----------------------------------------------------------
    t_clean = time.perf_counter()
    cleaned = get_rid_of(question_text)
    print("[Timer] Cleaning:", time.perf_counter() - t_clean, "sec")

    if not cleaned:
        return "I couldn't detect any movie titles."

    # ----------------------------------------------------------
    # 2) Movie extraction
    # ----------------------------------------------------------
    t_extract = time.perf_counter()
    parsed = extract_relations_and_entities(cleaned)
    entities = parsed.get("entity_candidates", [])
    movie_names = [e["text"].strip() for e in entities if len(e["text"].strip()) > 1]
    print("[Timer] Extraction:", time.perf_counter() - t_extract, "sec")

    if not movie_names:
        return "I couldn't detect any movie titles."

    # ----------------------------------------------------------
    # 3) Resolve → URIs
    # ----------------------------------------------------------
    t_resolve = time.perf_counter()
    movie_uris = []
    for name in movie_names:
        uri = find_uri_by_label(ent2lbl, name)
        if uri:
            movie_uris.append(uri)
            continue
        fallback_uri = fast_sbert_fallback(name, lbl2ent_type, sbert_model)
        if fallback_uri:
            movie_uris.append(fallback_uri)

    print("[Timer] Resolve URIs:", time.perf_counter() - t_resolve, "sec")

    if not movie_uris:
        return "I could not resolve any of these movies in the knowledge graph."

    # ----------------------------------------------------------
    # PART B — Metadata profile (kept in place)
    # ----------------------------------------------------------
    tB = time.perf_counter()
    metadata_summary = summarize_preferences(movie_uris, g)
    partB_sentence = generate_recommendation(metadata_summary)
    print("[Timer] Part B:", time.perf_counter() - tB, "sec")

    # ----------------------------------------------------------
    # PART D — Ultra-fast embedding recommender (PRIMARY)
    # ----------------------------------------------------------
    tD = time.perf_counter()
    try:
        emb_mids = [mid for mid in movie_uris if mid in combined_movie_embeddings_norm]

        if not emb_mids:
            user_vec = model.encode([cleaned])
            user_vec = user_vec / np.linalg.norm(user_vec)
            user_vec = np.pad(user_vec, ((0,0),(0,2)), mode='constant')
        else:
            M = np.vstack([combined_movie_embeddings_norm[mid] for mid in emb_mids])
            user_vec = np.mean(M, axis=0)
            user_vec /= np.linalg.norm(user_vec)
            user_vec = user_vec.reshape(1, -1)

        dist, inds = nn.kneighbors(user_vec, n_neighbors=100)
        recD_ids = [movie_ids[i] for i in inds[0]]
        recD_ids = [mid for mid in recD_ids if mid not in movie_uris][:5]
        recD_labels = []
        for mid in recD_ids:
            print(mid)
            uri = rdflib.term.URIRef(mid)
            lbl = ent2lbl.get(uri)   # safe lookup
            if lbl:                  # skip missing labels
                recD_labels.append(lbl)


        if recD_labels:
            partD_sentence = (
                "When our graph is checked these are similar movies: " +
                ", ".join(recD_labels) +
                "."
            )
        else:
            partD_sentence = ""

    except Exception as e:
        print("[Part D Error]", e)
        partD_sentence = ""

    print("[Timer] Part D:", time.perf_counter() - tD, "sec")

    # ----------------------------------------------------------
    # PART C — only if D fails
    # ----------------------------------------------------------
    partC_sentence = ""
    if not partD_sentence:
        print("[Running Part C — Part D was empty]")
        tC = time.perf_counter()

        metadata_signals = extract_rw_signals(metadata_summary)
        rw_candidates = biased_randomwalk_candidates(
            movie_uris=movie_uris,
            graph=g,
            ent2lbl=ent2lbl,
            metadata_signals=metadata_signals,
            max_neighbors=40
        )
        rw_scored = score_randomwalk_movies(
            liked_labels=movie_names,
            candidate_pairs=rw_candidates,
            sbert_model=sbert_model,
            top_k=3,
            min_score=0.3
        )
        partC_sentence = randomwalk_to_sentence(rw_scored)

        print("[Timer] Part C:", time.perf_counter() - tC, "sec")
    else:
        print("[Skipped] Part C — Part D already provided results.")

        # ----------------------------------------------------------
    # PART E — MovieLens (primary external recommender)
    # ----------------------------------------------------------
    tE = time.perf_counter()
    try:
        matched_ml, candidates_ml, scored_ml = recommend_movielens(movie_names)

        if matched_ml:
            top_genres = movielens_genre_profile(matched_ml)
            if top_genres:
                genre_sentence = (
                    "I get you like "
                    + ", ".join(top_genres) +
                    "."
                )
            else:
                genre_sentence = ""
        else:
            genre_sentence = ""
            scored_ml = []

        # ---- Helper functions ----
        def normalize_title(t: str) -> str:
            t = t.lower()
            t = re.sub(r"\(.*?\)", "", t)        # remove year/extra in parentheses
            t = re.sub(r"[^a-z0-9 ]", "", t)     # remove punctuation
            return t.strip()

        # Normalized versions of input movie names
        input_norms = [normalize_title(t) for t in movie_names]

        # Filter out MovieLens titles that are too similar to any input title
        filtered_ml = []
        for title, score in scored_ml:
            norm_t = normalize_title(title)

            too_close = any(
                fuzz.token_set_ratio(norm_t, inp) >= 90   # threshold can be tuned
                for inp in input_norms
            )

            if not too_close:
                filtered_ml.append(title)

        recommended_titles = filtered_ml[:3]

        if recommended_titles:
            rec_sentence = (
                "Furthermore, following movies also worth a look: "
                + ", ".join(recommended_titles) +
                "."
            )
        else:
            rec_sentence = ""

        partE_sentence = " ".join([genre_sentence, rec_sentence]).strip()

    except Exception as e:
        print("MovieLens error:", e)
        partE_sentence = ""
        rec_sentence = ""

    print("[Timer] Part E:", time.perf_counter() - tE, "sec")


    # ----------------------------------------------------------
    # PART A — runs ONLY IF MovieLens gives no recommendations
    # ----------------------------------------------------------
    partA_sentence = ""
    if not rec_sentence.strip():
        print("[Running Part A — MovieLens returned nothing]")
        tA = time.perf_counter()

        partA = recommend_fast_from_movies(
            movie_uris=movie_uris,
            graph=g,
            ent2lbl=ent2lbl,
            lbl2ent_type=lbl2ent_type,
            sbert_model=sbert_model,
            top_k=4
        )

        if partA:
            partA_sentence = (
                "Some movies you may enjoy include " +
                ", ".join([lbl for (lbl, uri, score) in partA[:3]]) +
                "."
            )
        print("[Timer] Part A:", time.perf_counter() - tA, "sec")
    else:
        print("[Skipped] Part A — MovieLens already gave results")

    # ----------------------------------------------------------
    # FINAL
    # ----------------------------------------------------------
    print("\n[Timer] TOTAL:", time.perf_counter() - t0, "sec")

    parts = [
        partB_sentence,
        partD_sentence,
        partC_sentence,
        partE_sentence,
        partA_sentence
    ]

    parts = [p for p in parts if p.strip()]

    if not parts:
        return "I could not generate recommendations."

    final_answer = " ".join(parts)
    print("[Recommender] Final:", final_answer)

    return final_answer


In [80]:
recommend_combined("Given that I like The Lion King, Pocahontas, and The Beauty and the Beast, can you recommend some movies?")


[Recommender] Raw question: Given that I like The Lion King, Pocahontas, and The Beauty and the Beast, can you recommend some movies?
[Timer] Cleaning: 0.0001544579999972484 sec

🔍 Entity 'The Lion King' disambiguation:
   Option 1: media franchise → the Disney The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.949)
   Option 2: dramatico-musical work → the musical The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.893)
   Option 3: animated film → the film The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.883)
   Option 4: film → the film The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.883)
   Option 5: video game → the video game The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.846)
   → Selected: the Disney The Lion King, Pocahontas, and The Beauty and the Beast, ? [Q196600]

🔍 Entity 'Po' disambiguation:
   Option 1: Q100663018 → The Lion King, the character and the Pocahontas, and The Beaut

/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


MovieLens error: Tensor for argument #2 'mat2' is on CPU, but expected it to be on GPU (while checking arguments for mm)
[Timer] Part E: 0.43597487499999943 sec
[Running Part A — MovieLens returned nothing]
[Timer] Part A: 0.03866641599998388 sec

[Timer] TOTAL: 2.3597941660000004 sec
[Recommender] Final: Adequate recommendations would be from the 1990s. When our graph is checked those are similar movies: Earth, La bête, 102 Dalmatians, Belle's Magical World, The Barbarians.


"Adequate recommendations would be from the 1990s. When our graph is checked those are similar movies: Earth, La bête, 102 Dalmatians, Belle's Magical World, The Barbarians."

In [81]:
print(ent2lbl.get(rdflib.term.URIRef('http://www.wikidata.org/entity/Q1138475')))

Katsuhisa Namase


In [82]:
def is_recommendation_question(q: str) -> bool:
    ql = q.lower()
    return any(k in ql for k in ["recommend", "similar to", "like ", "movies like", "suggest", "what should i watch", "suggest", "given that i like", "movie like"])

In [83]:
def select_type(message: str):
    """
    Detects whether the message requests a 'factual' or 'embedding' approach.
    Returns 'factual', 'embedding', or 'unknown'.
    """
    msg = message.lower().strip()

    if "factual approach" in msg or "with a factual" in msg:
        return "factual"
    elif "embedding approach" in msg or "with an embedding" in msg:
        return "embedding"
    elif is_recommendation_question(msg):
        return "recommendation"
    else:
        return "unknown"
    

In [ ]:
def on_question(q):
    """
    Main function to handle a user question q.
    Determines the answer type, extracts entities/relations,
    builds SPARQL queries, and fetches answers accordingly.
    """
    # Determine type and clean question
    answer_type = select_type(q)
    question = get_rid_of(q)

    # Extract entity & relation candidates
    parsed = extract_relations_and_entities(question)
    print(parsed["score"])
    ent_candidates = parsed.get("entity_candidates", [])
    rel_candidates = parsed.get("relation_candidates", [])

    print(f"\nClean question: {question}")
    print(f"Detected answer type: {answer_type}")
    print(f"Extracted entities: {ent_candidates}")
    print(f"Extracted relations: {rel_candidates}")

    # Build SPARQL queries
    queries = build_topk_sparql_queries(parsed, k=10)

    answers = []  # store all valid answers
    successful_query = False  # track if any query worked
    to_be_sent_to_bot = ""


    # Handle factual or embedding decision
    if answer_type == "factual":
        for qdict in queries:
            query_str = qdict.get("query")
            result = on_query(query_str, g)

            # check if query ran successfully and returned real answers
            # Handle both string error messages and actual results (which can be dates, numbers, etc.)
            is_error = (len(result) == 1 and 
                       isinstance(result[0], str) and 
                       "no matching results" in result[0].lower())
            if result and not is_error:
                successful_query = True
                # extend in case multiple results for this single query
                answers.extend(result)
                print(f"Query executed successfully. Found {len(result)} answers. Stopping after this query.")
                break  

        if successful_query:
            # Convert all answers to strings (handles dates, numbers, etc.)
            answer_strings = [str(a) for a in answers]
            # join all answers in a natural language form
            if len(answer_strings) == 1:
                answer_text = answer_strings[0]
            elif len(answer_strings) == 2:
                answer_text = f"{answer_strings[0]} and {answer_strings[1]}"
            else:
                answer_text = ", ".join(answer_strings[:-1]) + f", and {answer_strings[-1]}"

            print(f"The factual answer is: {answer_text}")
            to_be_sent_to_bot +=  f"The factual answer is: {answer_text}"

        else:
            print("I know you asked for a factual answer but your query seems quite complicated or has typos.")
            to_be_sent_to_bot += "I know you asked for a factual answer but your query seems quite complicated or has typos."
            print("So I'm switching to an embeddings answer:")
            to_be_sent_to_bot += " So I'm switching to an embeddings answer:"
            if ent_candidates and rel_candidates:
                try:
                    print(ent_candidates[0], rel_candidates[0])
                    df, embedding_text = embedding_answer(ent_candidates[0], rel_candidates[0])
                    print(df)
                    print(embedding_text)
                    to_be_sent_to_bot += f" {embedding_text}"
                except Exception as e:
                    print(f"Error occurred while getting embedding answer: {e}")
                    print("Missing entity or relation candidates for embedding reasoning.")
                    to_be_sent_to_bot += " Missing entity or relation candidates for embedding reasoning."

            else:
                print("Missing entity or relation candidates for embedding reasoning.")
                to_be_sent_to_bot += " Missing entity or relation candidates for embedding reasoning."

    elif answer_type == "embedding":
        if ent_candidates and rel_candidates:
            try:
                print("Embedding-based answer:")
                to_be_sent_to_bot += " Embedding-based answer:"

                found_valid = False
                # Try up to 5 relation candidates
                for i, rel_cand in enumerate(rel_candidates[:5]):
                    print(f"→ Trying relation candidate {i+1}: {rel_cand}")

                    result = embedding_answer(ent_candidates[0], rel_cand)
                    if result is None:
                        print(f"Could not find entity or relation for candidate {i+1}.")
                        continue

                    df, embedding_text = result

                    print(df)

                    # Skip if all labels are N/A
                    if df["Label"].eq("N/A").all():
                        print(f"All labels N/A for relation candidate {i+1}. Trying next one...")
                        continue

                    # ✅ Found a valid answer
                    print(df)
                    print(embedding_text)
                    to_be_sent_to_bot += f" {embedding_text}"
                    found_valid = True
                    break  # stop after first valid relation

                if not found_valid:
                    msg = "No valid embedding-based answer found after trying top 5 relation candidates."
                    print(msg)
                    to_be_sent_to_bot += f" {msg}"

            except Exception as e:
                print(f"Error occurred while getting embedding answer: {e}")
                to_be_sent_to_bot += " Missing entity or relation candidates for embedding reasoning."

        else:
            print("Missing entity or relation candidates for embedding reasoning.")
            to_be_sent_to_bot += " Missing entity or relation candidates for embedding reasoning."

    elif answer_type == "recommendation":
        
        try:
            # Use your full multi-part recommendation engine
            rec_text = recommend_combined(q)

            print(f"Recommendation answer: {rec_text}")
            to_be_sent_to_bot += rec_text

        except Exception as e:
            print(f"Error occurred while generating recommendation: {e}")
            fallback_msg = (
                "I could not generate a recommendation due to an internal issue."
            )
            print(fallback_msg)
            to_be_sent_to_bot += fallback_msg





    else:
        # If unknown type → try factual first, then embedding fallback
        for qdict in queries:
            query_str = qdict.get("query")
            result = on_query(query_str, g)

            # Handle both string error messages and actual results (which can be dates, numbers, etc.)
            is_error = (len(result) == 1 and 
                       isinstance(result[0], str) and 
                       "no matching results" in result[0].lower())
            if result and not is_error:
                successful_query = True
                answers.extend(result)
                print(f"Query executed successfully. Found {len(result)} answers. Stopping after this query.")
                break

        if successful_query:
            answer_strings = [str(a) for a in answers]
            # join all answers in a natural language form
            if len(answer_strings) == 1:
                answer_text = answer_strings[0]
            elif len(answer_strings) == 2:
                answer_text = f"{answer_strings[0]} and {answer_strings[1]}"
            else:
                answer_text = ", ".join(answer_strings[:-1]) + f", and {answer_strings[-1]}"

            print(f"The factual answer is: {answer_text}")
            to_be_sent_to_bot +=  f"The factual answer is: {answer_text}"

        if not successful_query:
            print("Your query seems quite complicated or has typos.")
            to_be_sent_to_bot += " Your query seems quite complicated or has typos."
            print("So I'm switching to an embeddings answer:")
            to_be_sent_to_bot += " So I'm switching to an embeddings answer:"
            if ent_candidates and rel_candidates:
                try:
                    result = embedding_answer(ent_candidates[0], rel_candidates[0])
                    if result is None:
                        print("Could not find entity or relation in the knowledge graph (possibly due to typos).")
                        to_be_sent_to_bot += " Could not find entity or relation in the knowledge graph (possibly due to typos)."
                    else:
                        df, embedding_text = result
                        print(df)
                        print(embedding_text)
                        to_be_sent_to_bot += f" {embedding_text}"
                except Exception as e:
                    print(f"Error occurred while getting embedding answer: {e}")
                    print("Could not find entity/relation for embedding reasoning.")
                    to_be_sent_to_bot += " Could not find entity/relation for embedding reasoning."
                    

            else:
                print("Could not find entity/relation for embedding reasoning.")
                to_be_sent_to_bot += " Could not find entity/relation for embedding reasoning."
    return to_be_sent_to_bot

In [85]:
on_question("Given that I like The Lion King, Pocahontas, and The Beauty and the Beast, can you recommend some movies?")


🔍 Entity 'The Lion King' disambiguation:
   Option 1: media franchise → the Disney The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.949)
   Option 2: dramatico-musical work → the musical The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.893)
   Option 3: animated film → the film The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.883)
   Option 4: film → the film The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.883)
   Option 5: video game → the video game The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.846)
   → Selected: the Disney The Lion King, Pocahontas, and The Beauty and the Beast, ? [Q196600]

🔍 Entity 'Po' disambiguation:
   Option 1: Q100663018 → The Lion King, the character and the Pocahontas, and The Beauty and the Beast, ? (score=0.978)
   → Selected: The Lion King, the character and the Pocahontas, and The Beauty and the Beast, ? [Q100663018]
entity_candidates: [{'text': 'The Lio

/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


"Detected recommendation question. Adequate recommendations would be from the 1990s. When our graph is checked those are similar movies: Earth, La bête, 102 Dalmatians, Belle's Magical World, The Barbarians."

In [87]:
import time

from speakeasypy import Chatroom, EventType, Speakeasy

DEFAULT_HOST_URL = 'https://speakeasy.ifi.uzh.ch'


class Agent:
    def __init__(self, username, password):
        self.username = username
        # Initialize the Speakeasy Python framework and login.
        self.speakeasy = Speakeasy(host=DEFAULT_HOST_URL, username=username, password=password)
        self.speakeasy.login()  # This framework will help you log out automatically when the program terminates.

        self.speakeasy.register_callback(self.on_new_message, EventType.MESSAGE)
        self.speakeasy.register_callback(self.on_new_reaction, EventType.REACTION)

    def listen(self):
        """Start listening for events."""
        self.speakeasy.start_listening()

    def on_new_message(self, message : str, room : Chatroom):
        """Callback function to handle new messages."""
        print(f"New message in room {room.room_id}: {message}")
        b = on_question(message)
        ans =  ", ".join(b)
        # Implement your agent logic here, e.g., respond to the message.
        # room.post_messages(f"Received your message: '{message}'")
        room.post_messages(f"{b}")

    def on_new_reaction(self, reaction : str, message_ordinal : int, room : Chatroom): 
        """Callback function to handle new reactions."""
        print(f"New reaction '{reaction}' on message #{message_ordinal} in room {room.room_id}")
        # Implement your agent logic here, e.g., respond to the reaction.
        room.post_messages(f"Thanks for your reaction: '{reaction}'")

    @staticmethod
    def get_time():
        return time.strftime("%H:%M:%S, %d-%m-%Y", time.localtime())


if __name__ == '__main__':
    demo_bot = Agent("MaroonRollingStone", "Rv2Iu8uT")
    demo_bot.listen()

Login successful. Session token: w0zlPyj8uTarY6ngmpZ2ockK6KN14F0y
on 0: 
on 0: 
on 0: 
on 0: 
on 0: 
on 0: 
on 0: 
on 1: New message in room 33dbc89c-24cc-4379-9410-27248345bb15: recommend me movies like beauty and the beast
on 1: entity_candidates: [] ranked_relations: [{'pid': 'p179', 'label': 'part of the series', 'category': 'TIME', 'semantic': 0.24391600489616394, 'lexical': 0.5555555555555556, 'boost': 0.1, 'score': 0.44418022467030416}, {'pid': 'p345', 'label': 'IMDb ID', 'category': 'PERSON', 'semantic': 0.3512316942214966, 'lexical': 0.4, 'boost': 0.1, 'score': 0.4356158471107483}, {'pid': 'p3275', 'label': 'storyboard artist', 'category': 'PERSON', 'semantic': 0.27940064668655396, 'lexical': 0.48484848484848486, 'boost': 0.1, 'score': 0.4336397172826709}, {'pid': 'p406', 'label': 'soundtrack release', 'category': 'TIME', 'semantic': 0.2855830788612366, 'lexical': 0.4666666666666666, 'boost': 0.1, 'score': 0.4294582060972849}, {'pid': 'p942', 'label': 'theme music', 'category'

on 3: /Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
        ret = a @ b
      /Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
        ret = a @ b
      /Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
        ret = a @ b


on 5: New message in room 33dbc89c-24cc-4379-9410-27248345bb15:  Recommend movies similar to Hamlet and Othello.
on 5: 
      🔍 Entity 'Hamlet' disambiguation:
on 5:    Option 1: Q116476516 → Similar to the Shakespeare Hamlet and Othello (score=0.957)
on 5:    Option 2: film → Similar to the film Hamlet and Othello (score=0.904)
on 5:    Option 3: film → Similar to the film Hamlet and Othello (score=0.904)
on 5:    Option 4: film → Similar to the film Hamlet and Othello (score=0.904)
on 5:    Option 5: film → Similar to the film Hamlet and Othello (score=0.904)
on 5:    → Selected: Similar to the Shakespeare Hamlet and Othello [Q116476516]
on 5: entity_candidates: [{'text': 'Hamlet', 'label': 'MISC', 'start': 11, 'end': 17, 'uri': ['http://www.wikidata.org/entity/Q41567', 'http://www.wikidata.org/entity/Q27178', 'http://www.wikidata.org/entity/Q1422824', 'http://www.wikidata.org/entity/Q2481391', 'http://www.wikidata.org/entity/Q898721'], 'type': ['Q116476516', 'Q11424', 'Q11424', 'Q11

on 5: /Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
        ret = a @ b
      /Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
        ret = a @ b
      /Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
        ret = a @ b


on 7: New message in room 33dbc89c-24cc-4379-9410-27248345bb15: Given that I like The Lion King, Pocahontas, and The Beauty and the Beast, can you recommend some movies?
on 7: 
      🔍 Entity 'The Lion King' disambiguation:
on 7:    Option 1: media franchise → the Disney The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.949)
on 7:    Option 2: dramatico-musical work → the musical The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.893)
on 7:    Option 3: animated film → the film The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.883)
on 7:    Option 4: film → the film The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.883)
on 7:    Option 5: video game → the video game The Lion King, Pocahontas, and The Beauty and the Beast, ? (score=0.846)
on 7:    → Selected: the Disney The Lion King, Pocahontas, and The Beauty and the Beast, ? [Q196600]
on 7: 
      🔍 Entity 'Po' disambiguation:
on 7:    Option 1: Q100663018 → The L

on 7: /Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
        ret = a @ b
      /Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
        ret = a @ b
      /Users/gizem/Documents/AI/Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
        ret = a @ b


on 9: 
Listening for events <●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●> 9 in 11:52.7 


KeyboardInterrupt: 